<div align="center">
  <img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6c/Javeriana.svg/3840px-Javeriana.svg.png?utm_source=es.wikipedia.org&utm_campaign=index&utm_content=thumbnail" alt="Pontificia Universidad Javeriana" width="300"/>
</div>

<h1 align="center">Taller de modelos de clasificación: predicción de salarios con Apache Spark MLlib</h1>

<table style="width:100%; border-collapse: collapse;" border="1">
<tr><td><strong>Estudiante</strong></td><td>Luis Ernesto Bueno Solano</td></tr>
<tr><td><strong>Universidad</strong></td><td>Pontificia Universidad Javeriana — Bogotá</td></tr>
<tr><td><strong>Facultad / Departamento</strong></td><td>Facultad de Ingeniería — Departamento de Ingeniería de Sistemas</td></tr>
<tr><td><strong>Curso</strong></td><td>Procesamiento de Datos a Gran Escala — código de clase 1055</td></tr>
<tr><td><strong>Profesor</strong></td><td>John Jairo Corredor Franco, PhD</td></tr>
<tr><td><strong>Plataforma</strong></td><td>Clúster de la universidad: Jupyter (10.195.34.34:8000) + Apache Spark 3.5 (PySpark, MLlib) + Hadoop HDFS</td></tr>
<tr><td><strong>Dataset</strong></td><td>Census Income (Adult) — UCI Machine Learning Repository</td></tr>
<tr><td><strong>Repositorio</strong></td><td><a href="https://github.com/Luis-bu/Procesamiento_Datos/tree/main/Taller_Salarios_MLlib">https://github.com/Luis-bu/Procesamiento_Datos/tree/main/Taller_Salarios_MLlib</a></td></tr>
<tr><td><strong>Fecha de entrega</strong></td><td>25-11-2026</td></tr>
</table>

## Resumen ejecutivo
Se construyeron, ajustaron y compararon modelos de clasificación de Spark MLlib para predecir si una persona gana más de 50 000 dólares al año. Se usaron los **48 842 registros** del censo de Estados Unidos de 1994 (dataset *Census Income* de UCI), cargados en HDFS y procesados con PySpark en el clúster del curso.

- **Datos:** el 23.9 % de las personas gana más de 50K. Se eliminaron 52 duplicados y 2 columnas sin valor predictivo. Los 3620 registros con faltantes (7.4 %) se conservaron como categoría `Desconocido`, porque ese grupo gana distinto al resto. Además, se agruparon las categorías raras y se aplicó logaritmo a las variables de capital.
- **Modelado:** se entrenaron 7 modelos base (regresión logística, árbol de decisión, Random Forest, GBT, Naive Bayes, Linear SVC y perceptrón multicapa). Los mejores parámetros de GBT y de la regresión logística se buscaron con validación cruzada, y el umbral de decisión se eligió con un conjunto de validación aparte, sin tocar los datos de prueba.
- **Resultado:** el mejor modelo es **GBT ajustado** (100 árboles de profundidad 4, umbral 0.30), con **AUC-ROC 0.917**, AUC-PR 0.788 y **F1 0.711** en prueba. Encuentra al **81 %** de las personas con más de 50K, frente al 63 % del mismo modelo con el umbral por defecto. El bootstrap confirma que su ventaja sobre Random Forest y la regresión logística es real, aunque pequeña frente a Random Forest.
- **Factores principales:** edad, nivel educativo, ocupación, rol en el hogar o estado civil, y ganancias y pérdidas de capital.
- **Implantación:** el modelo final quedó guardado en HDFS como `PipelineModel`; al recargarlo predice exactamente lo mismo sobre datos crudos.
- **Cumplimiento:** se cumplen los 4 criterios de éxito definidos antes de modelar. Como advertencia, el modelo detecta peor a las mujeres con ingresos altos que a los hombres (recall 0.73 frente a 0.82), algo que hay que tener en cuenta antes de usarlo en decisiones sobre personas.

## Abstract
This notebook builds, tunes and compares Spark MLlib classifiers that predict whether a person earns more than USD 50,000 per year, using the **48,842 records** of the UCI *Census Income* (Adult) dataset, extracted from the 1994 US Census. The data was stored in Hadoop HDFS and processed with PySpark on the course cluster.

After exploratory analysis and data cleaning, the steps were:
- Remove 52 duplicates and two non-predictive columns.
- Keep the 7.4 % of records with missing values as an `Unknown` category.
- Group rare categories and log-transform capital gains and losses.
- Build a feature `Pipeline` fitted on training data only.
- Train seven models (logistic regression, decision tree, random forest, gradient-boosted trees, naive Bayes, linear SVC and multilayer perceptron).
- Tune GBT and logistic regression with 3-fold `CrossValidator`, and pick the decision threshold on a held-out validation set.

The best model, **tuned GBT** (100 trees, depth 4, threshold 0.30), reaches **ROC-AUC 0.917, PR-AUC 0.788 and F1 0.711** on the test set, and finds 81 % of high earners. Bootstrap confidence intervals confirm that it outperforms random forest and logistic regression. Age, education, occupation, household role and capital gains are the strongest predictors. The final model was saved to HDFS as a `PipelineModel` and reloaded with identical predictions. A fairness check shows lower recall for women than for men (0.73 vs 0.82), which should be addressed before using the model for decisions about people.

## Contexto
El ingreso de una persona depende de factores como su edad, su nivel educativo, su ocupación, las horas que trabaja y su situación familiar. Predecir si alguien gana más o menos de cierto umbral sirve, por ejemplo, para segmentar clientes en un banco, focalizar programas sociales o estudiar brechas salariales.

Se usa el dataset **Census Income** (también llamado *Adult*) del repositorio UCI Machine Learning. Barry Becker lo extrajo de la base de datos del **censo de Estados Unidos de 1994**, y Ron Kohavi lo publicó en 1996. Tiene **48 842 registros** y **14 variables** sobre la persona (edad, tipo de empleo, educación, estado civil, ocupación, raza, sexo, ganancias y pérdidas de capital, horas de trabajo por semana y país de origen), más la variable objetivo: si el ingreso anual supera los **50 000 dólares** (`>50K`) o no (`<=50K`).

El dataset viene en dos archivos, `adult.data` (32 561 registros) y `adult.test` (16 281 registros), que se suben al HDFS del clúster y se procesan con PySpark.

### Comprensión del negocio (CRISP-DM)
**Pregunta de negocio:** ¿se puede estimar, solo con datos demográficos y laborales, si una persona gana más de 50 000 dólares al año, y qué tan confiable es esa estimación?

**Usos posibles del modelo:**
- *Banca y mercadeo:* ofrecer productos (tarjetas, créditos, inversiones) a quienes probablemente tienen ingresos altos. Aquí importa la **precisión**, porque cada oferta mal dirigida cuesta.
- *Programas sociales:* identificar a quienes probablemente tienen ingresos bajos. Aquí importa no dejar a nadie por fuera: el **recall** de la clase de interés.
- *Estudios de política pública:* entender qué variables se asocian con el ingreso y dónde hay brechas (por sexo, raza o educación).

**Criterios de éxito definidos antes de modelar:**
1. Superar la línea base de predecir siempre la clase mayoritaria (`<=50K`, alrededor del 76 % de accuracy).
2. AUC-ROC ≥ 0.90 en el conjunto de prueba.
3. F1 ≥ 0.65 en la clase `>50K`.
4. Tiempos de entrenamiento manejables en el clúster (segundos o pocos minutos por modelo).

El trabajo sigue las fases de CRISP-DM: comprensión del negocio, comprensión de los datos (EDA), preparación de los datos, modelado, evaluación e implantación (en este taller, guardar el modelo en HDFS para reutilizarlo).

*Fuente:* Kohavi, R. (1996). *Census Income* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5GP7S

## Objetivos
**General**
- Construir, ajustar y comparar modelos de clasificación de Spark MLlib que predigan si una persona gana más de 50 000 dólares al año a partir de sus datos del censo.

**Específicos**
1. Cargar el dataset en Hadoop HDFS y leerlo con PySpark.
2. Realizar un análisis exploratorio (EDA) para entender las variables, su calidad y su relación con el ingreso.
3. Diagnosticar los problemas de calidad de los datos con evidencia (tablas y gráficas) y tomar una decisión justificada para cada uno.
4. Implementar el tratamiento de los datos y un `Pipeline` de características que se ajuste solo con datos de entrenamiento.
5. Entrenar siete modelos de MLlib (regresión logística, árbol de decisión, Random Forest, GBT, Naive Bayes, Linear SVC y perceptrón multicapa) y ajustar los hiperparámetros de los más prometedores con validación cruzada.
6. Comparar los modelos con varias métricas, curvas, intervalos de confianza y costo de entrenamiento, y elegir el más adecuado.
7. Guardar el modelo final en HDFS y formular conclusiones y recomendaciones de uso.

## Contenido
| Sección | Criterio de la rúbrica |
|---|---|
| Presentación, resumen, contexto y objetivos | Presentación, identificación, objetivos, contexto |
| 1. Carga de datos en HDFS | Presentación (uso de la plataforma) |
| 2. EDA general | EDA general |
| 3. Preparación de datos: análisis y gráficas | Preparación de datos: análisis, gráficas |
| 4. Tratamiento de datos y pipeline | Tratamiento de datos y pipeline |
| 5. Aplicación de modelos MLlib | Aplicación de modelos de MLlib |
| 6. Comparativa general de modelos | Comparativa general de modelos |
| 7. Conclusiones y recomendaciones | Conclusiones, recomendaciones |
| 8. English summary + repositorio | Resumen, repositorio, inglés |

## Librerías y sesión de Spark

In [ ]:
#Librerías generales
import numpy as np
import pandas as pd
import time

#Apache Spark
# findspark busca la instalación de Spark en la ruta indicada y la agrega al path de Python.
# Debe ir ANTES de importar pyspark
import findspark
findspark.init('/mnt/sda1/Cluster/Spark')

import pyspark
from pyspark import SparkConf
from pyspark.sql import SparkSession
# F = alias de las funciones de columna: F.col, F.when, F.count, F.avg...
from pyspark.sql import functions as F
# Tipos para definir el esquema del archivo a mano
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

#Spark ML: transformaciones, pipeline, modelos, ajuste y evaluadores
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier, RandomForestClassifier,
                                       GBTClassifier, NaiveBayes, LinearSVC, MultilayerPerceptronClassifier)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

#Gráficas y métricas auxiliares
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score

In [ ]:
#Sesión Spark
configura = SparkConf()
configura.setAppName("BuenoSalario_1055")

# builder -> config(conf=...) -> getOrCreate(): crea la sesión o devuelve la que ya existe
sparkLuis = SparkSession.builder.config(conf=configura).getOrCreate()
# SparkContext asociado a la sesión (API de RDD)
sparkContextoLuis = sparkLuis.sparkContext
# Solo se muestran errores, no advertencias
sparkContextoLuis.setLogLevel("ERROR")
print("Versión de Spark:", sparkLuis.version)
sparkLuis

# **1. Carga de datos en HDFS**
### 1.1 Subida del dataset al data lake
El dataset no está en la carpeta `/csv` del clúster, así que se sube a la carpeta personal en HDFS:
1. Descargar el zip desde https://archive.ics.uci.edu/dataset/20/census+income y descomprimirlo.
2. Subir `adult.data` y `adult.test` a la carpeta de trabajo de Jupyter, la misma de este cuaderno (botón **Upload**).
3. Crear la carpeta `Salario` dentro de la carpeta personal en HDFS y copiar ahí los dos archivos.

In [ ]:
# Se crea la carpeta del taller dentro de la carpeta personal (-p: no falla si ya existe)
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -mkdir -p /Formatos1055/BuenoLuis/Salario

In [ ]:
# Se copian los dos archivos desde Jupyter a HDFS (-f: sobrescribe si ya estaban)
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -put -f adult.data adult.test /Formatos1055/BuenoLuis/Salario

In [ ]:
# Se revisa que los archivos estén en HDFS
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -ls /Formatos1055/BuenoLuis/Salario

### 1.2 Lectura con PySpark
Los archivos de UCI tienen varias particularidades que se resuelven al leerlos:
- **No tienen encabezado**: los nombres de las columnas se definen a mano en un esquema (`StructType`), que además fija el tipo de cada columna. Así Spark no tiene que recorrer el archivo para adivinar los tipos (`inferSchema`). Las columnas se nombran de una vez en español.
- **Los valores van separados por coma y espacio** (`39, State-gov, ...`): `ignoreLeadingWhiteSpace` quita ese espacio.
- **Los faltantes vienen como `?`**: con `nullValue` se leen directamente como nulos.
- **`adult.test` empieza con la línea `|1x3 Cross validator`**: con `comment = "|"` Spark la ignora.

In [ ]:
# Esquema: nombre, tipo y si admite nulos (True) para cada una de las 15 columnas
esquema = StructType([
    StructField("Edad", IntegerType(), True),
    StructField("ClaseTrabajo", StringType(), True),
    StructField("PesoFinal", IntegerType(), True),
    StructField("Educacion", StringType(), True),
    StructField("NumEducacion", IntegerType(), True),
    StructField("EstadoCivil", StringType(), True),
    StructField("Ocupacion", StringType(), True),
    StructField("Relacion", StringType(), True),
    StructField("Raza", StringType(), True),
    StructField("Sexo", StringType(), True),
    StructField("GananciaCapital", IntegerType(), True),
    StructField("PerdidaCapital", IntegerType(), True),
    StructField("HorasSemana", IntegerType(), True),
    StructField("PaisOrigen", StringType(), True),
    StructField("Ingreso", StringType(), True),
])

def leerAdult(ruta):
    return (sparkLuis.read.format("csv")
            .schema(esquema)
            .option("ignoreLeadingWhiteSpace", "true")
            .option("nullValue", "?")
            .option("comment", "|")
            .load(ruta))

urlData = "hdfs://10.195.34.34:9000/Formatos1055/BuenoLuis/Salario/adult.data"
urlTest = "hdfs://10.195.34.34:9000/Formatos1055/BuenoLuis/Salario/adult.test"
df00 = leerAdult(urlData)
df01 = leerAdult(urlTest)

print(f"adult.data: {df00.count()} registros")
print(f"adult.test: {df01.count()} registros")
df00.show(5)

In [ ]:
# Etiquetas de cada archivo: en adult.test terminan en punto
df00.groupBy("Ingreso").count().show()
df01.groupBy("Ingreso").count().show()

**Lectura.** `adult.data` trae 32 561 registros y `adult.test` 16 281: en total 48 842, los mismos que reporta UCI. El esquema dejó cada columna con su tipo (enteros para edad, horas y capital; texto para las categóricas). En `adult.test` las etiquetas terminan en punto (`<=50K.` y `>50K.`): si se unieran los archivos así, quedarían cuatro clases en lugar de dos, por eso se limpian al unirlos.

### 1.3 Unión de los dos archivos y variable objetivo
UCI entrega los datos ya divididos en entrenamiento y prueba, pero en este taller la división se hace después, con `randomSplit`, sobre los datos ya limpios. Por eso se unen los dos archivos, se quita el punto final de las etiquetas de `adult.test` y se crea `Etiqueta` (**1** = `>50K`, **0** = `<=50K`), que MLlib necesita en formato numérico.

In [ ]:
# unionByName: une por nombre de columna (las dos tablas tienen el mismo esquema)
# regexp_replace: borra el punto final de "<=50K." y ">50K."
df02 = (df00.unionByName(df01)
            .withColumn("Ingreso", F.regexp_replace("Ingreso", r"\.", ""))
            .withColumn("Etiqueta", (F.col("Ingreso") == ">50K").cast("int")))
totalReg = df02.count()
print(f"Total de registros: {totalReg}  |  Columnas: {len(df02.columns)}")
df02.groupBy("Ingreso", "Etiqueta").count().show()

# **2. EDA general**
Análisis exploratorio sobre los datos tal como llegan (antes de limpiar): estructura, variable objetivo, estadísticas, categorías, relación de cada variable con el ingreso, correlaciones e interacciones. Los cálculos pesados se hacen en Spark (`groupBy` + `agg`) y solo los resultados, que son pequeños, se pasan a pandas para graficar.

### 2.1 Diccionario de datos
| Columna original | Nombre en el taller | Tipo | Descripción |
|---|---|---|---|
| `age` | `Edad` | numérica | Edad en años |
| `workclass` | `ClaseTrabajo` | categórica | Tipo de empleador: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked |
| `fnlwgt` | `PesoFinal` | numérica | Peso muestral del censo: cuántas personas de la población representa el registro |
| `education` | `Educacion` | categórica | Máximo nivel educativo alcanzado (16 niveles) |
| `education-num` | `NumEducacion` | numérica | El mismo nivel educativo como número del 1 (Preschool) al 16 (Doctorate) |
| `marital-status` | `EstadoCivil` | categórica | Married-civ-spouse, Never-married, Divorced, Separated, Widowed, ... |
| `occupation` | `Ocupacion` | categórica | Ocupación (14 tipos): Prof-specialty, Exec-managerial, Sales, ... |
| `relationship` | `Relacion` | categórica | Rol en el hogar: Husband, Wife, Own-child, Not-in-family, Unmarried, Other-relative |
| `race` | `Raza` | categórica | White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other |
| `sex` | `Sexo` | categórica | Male / Female |
| `capital-gain` | `GananciaCapital` | numérica | Ganancias de capital en el año (dólares) |
| `capital-loss` | `PerdidaCapital` | numérica | Pérdidas de capital en el año (dólares) |
| `hours-per-week` | `HorasSemana` | numérica | Horas trabajadas por semana |
| `native-country` | `PaisOrigen` | categórica | País de origen (41 países) |
| `income` | `Ingreso` / `Etiqueta` | binaria | **Variable objetivo**: `>50K` (1) o `<=50K` (0) |

### 2.2 Estructura y variable objetivo

In [ ]:
df02.printSchema()

pdObjetivo = df02.groupBy("Ingreso").count().withColumn("Porcentaje", F.round(F.col("count")/totalReg*100, 2)).toPandas()
print(pdObjetivo)
plt.figure(figsize=(6, 4))
sns.barplot(data=pdObjetivo, x="Ingreso", y="count", order=["<=50K", ">50K"], color="steelblue")
for i, fila in enumerate(pdObjetivo.set_index("Ingreso").loc[["<=50K", ">50K"]].itertuples()):
    plt.text(i, fila.count, f"{fila.Porcentaje} %", ha="center", va="bottom")
plt.title("Distribución de la variable objetivo")
plt.ylabel("Personas")
plt.show()

**Lectura.** El 76.07 % de las personas gana `<=50K` y el 23.93 % (11 687) gana `>50K`. Las clases están desbalanceadas, aunque no de forma extrema. Esto define el piso de comparación: un modelo que diga siempre `<=50K` ya acierta el 76 %, así que el accuracy por sí solo no basta para juzgar un modelo. Por eso se evaluará también con AUC, precisión, recall y F1 de la clase `>50K`.

### 2.3 Estadística descriptiva de las variables numéricas

In [ ]:
df02.describe("Edad", "PesoFinal", "NumEducacion", "GananciaCapital", "PerdidaCapital", "HorasSemana").show()
print("GananciaCapital > 0  :", df02.where("GananciaCapital > 0").count())
print("GananciaCapital 99999:", df02.where("GananciaCapital = 99999").count())
print("PerdidaCapital > 0   :", df02.where("PerdidaCapital > 0").count())

**Lectura.** 
- **Edad**: de 17 a 90 años, promedio 38.6.
- **NumEducacion**: promedio 10.1, que equivale a *Some-college* (algo de universidad sin título).
- **HorasSemana**: promedio 40.4, entre 1 y 99 horas.
- **GananciaCapital**: solo 4035 personas (8.3 %) tienen un valor distinto de cero, y 244 tienen exactamente 99 999. Ese valor es el tope con el que el censo codifica las ganancias muy altas, no una cifra exacta.
- **PerdidaCapital**: solo 2282 personas (4.7 %) tienen un valor distinto de cero.
- **PesoFinal**: va de 12 285 a 1 490 400. No es una característica de la persona, sino cuántas personas de la población representa cada registro.

### 2.4 Variables categóricas

In [ ]:
# Para cada columna categórica: cuántas personas hay en cada categoría y qué porcentaje gana >50K
def resumenCategoria(df, columna, n=20):
    df.groupBy(columna).agg(
        F.count("*").alias("Personas"),
        F.round(F.avg("Etiqueta")*100, 1).alias("Pct_mayor50K")
    ).orderBy(F.desc("Personas")).show(n, truncate=False)

for c in ["ClaseTrabajo", "EstadoCivil", "Ocupacion", "Relacion", "Raza", "Sexo"]:
    resumenCategoria(df02, c)
print("Países distintos:", df02.select(F.countDistinct("PaisOrigen")).first()[0])
resumenCategoria(df02, "PaisOrigen", 10)

**Lectura.** 
- **ClaseTrabajo**: el 69.4 % trabaja en el sector privado (21.8 % gana >50K). Los independientes con empresa constituida (`Self-emp-inc`) tienen la tasa más alta (55.3 %). `Without-pay` y `Never-worked` suman solo 31 personas.
- **EstadoCivil**: la diferencia es enorme. Los casados que viven con su cónyuge (`Married-civ-spouse`) llegan al 44.6 % y los solteros (`Never-married`) apenas al 4.5 %.
- **Ocupacion**: `Exec-managerial` (47.8 %) y `Prof-specialty` (45.1 %) contra `Priv-house-serv` (1.2 %) y `Other-service` (4.1 %).
- **Relacion**: `Husband` (44.9 %) y `Wife` (46.9 %) contra `Own-child` (1.5 %), que son sobre todo jóvenes que viven con sus padres.
- **Sexo**: 30.4 % de los hombres gana >50K frente a 10.9 % de las mujeres.
- **PaisOrigen**: 41 países, pero el 89.7 % es de Estados Unidos. El segundo es México con 951 personas; el resto tiene muy pocos registros cada uno.

### 2.5 Relación de cada variable con el ingreso

In [ ]:
# Porcentaje de personas con >50K por categoría, calculado en Spark y devuelto como pandas.
# coalesce: para graficar, los nulos se muestran como "(faltante)"
def tasaMayor50K(df, columna):
    return (df.withColumn(columna, F.coalesce(F.col(columna).cast("string"), F.lit("(faltante)")))
              .groupBy(columna)
              .agg(F.count("*").alias("Personas"), F.round(F.avg("Etiqueta")*100, 1).alias("Pct_mayor50K"))
              .orderBy(columna)
              .toPandas())

# Línea de referencia: porcentaje general de >50K
pctGeneral = df02.select(F.avg("Etiqueta")).first()[0]*100
print(f"Porcentaje general con >50K: {round(pctGeneral, 1)} %")

# Columnas numéricas en pandas (48 mil filas, 7 columnas: cabe sin problema en el driver)
pdNum = df02.select("Edad", "PesoFinal", "NumEducacion", "GananciaCapital", "PerdidaCapital",
                    "HorasSemana", "Etiqueta", "Ingreso").toPandas()

In [ ]:
# Edad y horas según ingreso
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
# common_norm=False: cada grupo suma 1 por separado, así se comparan las formas aunque un grupo sea 3 veces más grande
sns.histplot(data=pdNum, x="Edad", hue="Ingreso", stat="density", common_norm=False,
             bins=30, hue_order=["<=50K", ">50K"], ax=ejes[0])
ejes[0].set_title("Distribución de la edad según ingreso")
sns.boxplot(data=pdNum, x="Ingreso", y="HorasSemana", order=["<=50K", ">50K"], ax=ejes[1])
ejes[1].set_title("Horas trabajadas por semana según ingreso")
plt.tight_layout()
plt.show()
pdNum.groupby("Ingreso")[["Edad", "HorasSemana"]].agg(["mean", "median"]).round(1)

**Lectura.** Las personas con >50K son mayores: tienen en promedio 44.3 años, frente a 36.9 de las que ganan `<=50K`. Por debajo de 25 años casi nadie supera los 50K, y el grupo de >50K se concentra entre los 35 y los 55 años. En horas, la mediana es 40 en los dos grupos, pero el grupo de >50K trabaja más en promedio (45.5 horas frente a 38.8): la mitad central de ese grupo trabaja entre 40 y 50 horas, contra 35 a 40 del otro.

In [ ]:
# Nivel educativo
pdEdu = tasaMayor50K(df02.withColumn("NumEducacion", F.lpad(F.col("NumEducacion").cast("string"), 2, "0")), "NumEducacion")
plt.figure(figsize=(12, 5))
sns.barplot(data=pdEdu, x="NumEducacion", y="Pct_mayor50K", color="steelblue")
plt.axhline(pctGeneral, color="red", linestyle="--", label=f"General ({round(pctGeneral, 1)} %)")
plt.title("% de personas con ingreso >50K según nivel educativo (01 = Preschool ... 16 = Doctorate)")
plt.ylabel("% con >50K")
plt.legend()
plt.show()
pdEdu.T

**Lectura.** La educación es una de las variables con mayor efecto. Hasta el nivel 08 (12th, sin graduarse de secundaria) menos del 8 % gana >50K. Con secundaria completa (`HS-grad`, 09) sube a 15.9 %. Desde los grados técnicos (11–12) se supera el promedio general, y luego la tasa crece rápido: 41.3 % con pregrado (`Bachelors`, 13), 54.9 % con maestría, 74.0 % en escuelas profesionales (medicina, derecho) y 72.6 % con doctorado.

In [ ]:
# Estado civil y relación en el hogar
pdCivil = tasaMayor50K(df02, "EstadoCivil").sort_values("Pct_mayor50K", ascending=False)
pdRel = tasaMayor50K(df02, "Relacion").sort_values("Pct_mayor50K", ascending=False)

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=pdCivil, y="EstadoCivil", x="Pct_mayor50K", color="steelblue", ax=ejes[0])
ejes[0].set_title("% con >50K según estado civil")
sns.barplot(data=pdRel, y="Relacion", x="Pct_mayor50K", color="darkorange", ax=ejes[1])
ejes[1].set_title("% con >50K según relación en el hogar")
for eje in ejes:
    eje.axvline(pctGeneral, color="red", linestyle="--")
    eje.set_xlabel("% con >50K")
plt.tight_layout()
plt.show()

**Lectura.** Las dos gráficas muestran lo mismo desde dos ángulos: vivir con el cónyuge es lo que más se asocia a ganar >50K (`Married-civ-spouse` 44.6 %, `Wife` 46.9 %, `Husband` 44.9 %). `Married-AF-spouse` aparece alto (37.8 %), pero solo tiene 37 personas. Todos los demás grupos están por debajo del 11 %. Parte de este efecto es la edad: los solteros (`Never-married`, 4.5 %) y los hijos que viven en casa (`Own-child`, 1.5 %) son en su mayoría jóvenes. Además, `EstadoCivil` y `Relacion` dicen casi lo mismo, y `Husband` / `Wife` también codifican el sexo. Esa redundancia importa para Naive Bayes y para el análisis de equidad.

In [ ]:
# Ocupación (los nulos aparecen como "(faltante)")
pdOcu = tasaMayor50K(df02, "Ocupacion").sort_values("Pct_mayor50K", ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=pdOcu, y="Ocupacion", x="Pct_mayor50K", color="steelblue")
plt.axvline(pctGeneral, color="red", linestyle="--", label="General")
plt.title("% con >50K según ocupación")
plt.xlabel("% con >50K")
plt.legend()
plt.show()

**Lectura.** Los cargos directivos (`Exec-managerial`, 47.8 %) y los profesionales (`Prof-specialty`, 45.1 %) son los únicos cerca del 50 %. `Armed-Forces` aparece alto (33.3 %), pero solo tiene 15 personas. En el otro extremo están el servicio doméstico (`Priv-house-serv`, 1.2 %), otros servicios (4.1 %) y los ayudantes de limpieza y carga (6.7 %). Las personas **sin ocupación registrada** (`(faltante)`) quedan en 9.4 %, muy por debajo del promedio. Este dato se retoma en la sección 3.

In [ ]:
# Sexo, raza y ganancias de capital
pdSexo = tasaMayor50K(df02, "Sexo")
pdRaza = tasaMayor50K(df02, "Raza").sort_values("Pct_mayor50K", ascending=False)
pdCapital = tasaMayor50K(df02.withColumn("TieneGanancia", F.when(F.col("GananciaCapital") > 0, "Sí").otherwise("No")),
                         "TieneGanancia")

fig, ejes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.barplot(data=pdSexo, x="Sexo", y="Pct_mayor50K", color="steelblue", ax=ejes[0])
ejes[0].set_title("% con >50K según sexo")
sns.barplot(data=pdRaza, x="Pct_mayor50K", y="Raza", color="slategray", ax=ejes[1])
ejes[1].set_title("% con >50K según raza")
sns.barplot(data=pdCapital, x="TieneGanancia", y="Pct_mayor50K", color="seagreen", ax=ejes[2])
ejes[2].set_title("% con >50K según si tiene ganancias de capital")
ejes[0].axhline(pctGeneral, color="red", linestyle="--")
ejes[1].axvline(pctGeneral, color="red", linestyle="--")
ejes[2].axhline(pctGeneral, color="red", linestyle="--")
plt.tight_layout()
plt.show()
print(pdSexo, "\n", pdCapital)

**Lectura.** 
- **Sexo**: los hombres superan los 50K casi tres veces más que las mujeres (30.4 % frente a 10.9 %), un reflejo de la brecha salarial del censo de 1994.
- **Raza**: `Asian-Pac-Islander` (26.9 %) y `White` (25.4 %) están por encima del promedio, y `Black` (12.1 %), `Amer-Indian-Eskimo` (11.7 %) y `Other` (12.3 %) por debajo.
- **Ganancias de capital** (inversiones, venta de activos): el 61.7 % de quienes las tienen gana >50K, contra 20.5 % de quienes no.

Sexo y raza son atributos sensibles. Por eso, en la sección 6.8 se revisa si el modelo final se equivoca de forma distinta según el grupo.

### 2.6 Correlaciones entre variables numéricas

In [ ]:
# Correlación de Pearson entre numéricas y la etiqueta
correlaciones = pdNum.drop(columns="Ingreso").corr().round(3)
plt.figure(figsize=(8, 6))
sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Matriz de correlación (Pearson)")
plt.show()
correlaciones["Etiqueta"].sort_values(ascending=False)

**Lectura.** 
- **Relación con la etiqueta:** la correlación más alta es la de `NumEducacion` (0.33), seguida de `Edad` (0.23), `HorasSemana` (0.23), `GananciaCapital` (0.22) y `PerdidaCapital` (0.15). Son correlaciones moderadas: ninguna variable por sí sola separa bien las clases, y por eso hace falta combinarlas en un modelo.
- **`PesoFinal`:** su correlación con la etiqueta es prácticamente 0 (−0.006).
- **Entre las variables predictoras:** la correlación más alta es 0.14 (educación con horas), así que no hay variables numéricas fuertemente correlacionadas que haya que eliminar.

Pearson solo mide relaciones lineales entre numéricas. La redundancia más fuerte del dataset está en las categóricas (`EstadoCivil`, `Relacion` y `Sexo`), como se vio en 2.5.

### 2.7 Interacción entre edad y educación

In [ ]:
# Porcentaje con >50K cruzando grupo de edad y nivel educativo (calculado en Spark)
dfInter = (df02
    .withColumn("GrupoEdad", F.when(F.col("Edad") < 25, "17-24").when(F.col("Edad") < 35, "25-34")
                              .when(F.col("Edad") < 45, "35-44").when(F.col("Edad") < 55, "45-54")
                              .when(F.col("Edad") < 65, "55-64").otherwise("65+"))
    .withColumn("NivelEducativo", F.when(F.col("NumEducacion") <= 8, "1. Sin secundaria")
                                   .when(F.col("NumEducacion") == 9, "2. Secundaria")
                                   .when(F.col("NumEducacion") <= 12, "3. Técnico / algo de univ.")
                                   .when(F.col("NumEducacion") == 13, "4. Pregrado")
                                   .otherwise("5. Posgrado")))
pdInter = (dfInter.groupBy("NivelEducativo", "GrupoEdad")
                  .agg(F.round(F.avg("Etiqueta")*100, 1).alias("Pct"))
                  .toPandas()
                  .pivot(index="NivelEducativo", columns="GrupoEdad", values="Pct"))
plt.figure(figsize=(10, 5))
sns.heatmap(pdInter, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "% con >50K"})
plt.title("% con ingreso >50K por nivel educativo y grupo de edad")
plt.show()

**Lectura.** En todos los niveles educativos la tasa de >50K crece con la edad hasta los 45–54 años y luego baja. Pero el efecto de la educación no se suma igual en todas las edades. Entre los 17 y 24 años casi nadie gana >50K, ni siquiera con posgrado (10.8 %). Entre los 45 y 54, en cambio, la diferencia es enorme: 72.4 % con posgrado frente a 10.1 % sin secundaria. Esto es una **interacción**: el efecto de una variable depende del valor de otra. Los modelos lineales no la capturan por sí solos, mientras que los modelos de árboles sí, lo que anticipa que estos últimos van a rendir mejor.

### 2.8 Resumen de hallazgos del EDA
| # | Hallazgo | Evidencia | Implicación |
|---|---|---|---|
| 1 | Clases desbalanceadas | 76.1 % `<=50K` / 23.9 % `>50K` | Evaluar con AUC, F1 y recall, no solo accuracy; revisar el umbral |
| 2 | La educación es la variable numérica más asociada | Correlación 0.33; de 1.2 % (Preschool) a 74 % (Prof-school) | Variable clave |
| 3 | Vivir con el cónyuge se asocia fuertemente al ingreso alto | 44.6 % casados vs. 4.5 % solteros | `EstadoCivil` y `Relacion` son redundantes entre sí |
| 4 | La edad interactúa con la educación | 10.8 % (posgrado, 17–24) vs. 72.4 % (posgrado, 45–54) | Los modelos de árboles deberían rendir mejor |
| 5 | Ocupación y capital marcan grandes diferencias | Directivos 47.8 % vs. servicio doméstico 1.2 %; con ganancias de capital 61.7 % | Variables útiles |
| 6 | Brechas por sexo y raza | Hombres 30.4 % vs. mujeres 10.9 % | Atributos sensibles: revisar la equidad del modelo |
| 7 | Problemas de calidad | Faltantes, duplicados, columnas redundantes, categorías raras, capital muy asimétrico | Se analizan en la sección 3 |

# **3. Preparación de datos: análisis y gráficas**
Diagnóstico de los problemas de calidad, con evidencia para cada decisión. Aquí **no se modifica** ningún dato; las transformaciones se aplican en la sección 4.

### 3.1 Valores faltantes

In [ ]:
# Nulos por columna: isNull() da True/False; cast("int") lo vuelve 1/0 y sum() los cuenta
pdNulos = df02.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df02.columns]).toPandas().T
pdNulos.columns = ["Nulos"]
pdNulos["Porcentaje"] = (pdNulos["Nulos"]/totalReg*100).round(2)
pdNulos = pdNulos[pdNulos["Nulos"] > 0].sort_values("Nulos", ascending=False)
print(pdNulos)
print("Registros con al menos un faltante:", totalReg - df02.na.drop().count())
print("ClaseTrabajo y Ocupacion nulos a la vez:",
      df02.where(F.col("ClaseTrabajo").isNull() & F.col("Ocupacion").isNull()).count())

# ¿Los que tienen faltantes ganan distinto al resto?
pdFaltaOcu = (df02.groupBy(F.when(F.col("Ocupacion").isNull(), "Ocupación faltante").otherwise("Ocupación registrada").alias("Grupo"))
                  .agg(F.count("*").alias("Personas"), F.round(F.avg("Etiqueta")*100, 2).alias("Pct_mayor50K")).toPandas())
print(pdFaltaOcu)

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x=pdNulos.index, y=pdNulos["Porcentaje"], color="indianred", ax=ejes[0])
ejes[0].set_title("% de valores faltantes por columna")
ejes[0].set_ylabel("% de registros")
ejes[0].set_xlabel("")
sns.barplot(data=pdFaltaOcu, x="Grupo", y="Pct_mayor50K", color="steelblue", ax=ejes[1])
ejes[1].axhline(pctGeneral, color="red", linestyle="--")
ejes[1].set_title("% con >50K según si falta la ocupación")
plt.tight_layout()
plt.show()

**Lectura.** 
- **Dónde están:** solo tres columnas tienen faltantes: `Ocupacion` (2809, 5.75 %), `ClaseTrabajo` (2799, 5.73 %) y `PaisOrigen` (857, 1.75 %). En total son **3620 registros con al menos un faltante (7.4 %)**.
- **Coinciden entre sí:** las 2799 personas sin `ClaseTrabajo` tampoco tienen `Ocupacion`. Las otras 10 sin ocupación son de la clase `Never-worked`, para quienes la ocupación no aplica.
- **No son aleatorios:** las personas sin ocupación registrada ganan >50K mucho menos que el resto (9.43 % frente a 24.81 %). El faltante dice algo sobre la persona.

**Decisión:** no borrarlos (se perdería el 7.4 % de los datos y un patrón real) ni imputarlos con la moda (se mezclaría a este grupo con el sector privado, que tiene 21.8 %). Se reemplazan por la categoría `Desconocido`.

### 3.2 Registros duplicados

In [ ]:
duplicados = totalReg - df02.dropDuplicates().count()
print(f"Registros duplicados (idénticos en todas las columnas): {duplicados} ({round(duplicados/totalReg*100, 2)} %)")

**Lectura.** Hay 52 registros idénticos en las 15 columnas originales, incluido `PesoFinal`, que es casi único por persona. Que coincidan en todo, incluido ese peso, indica que son el mismo registro capturado dos veces. **Decisión:** eliminarlos (0.11 % de los datos), para que ninguna persona cuente doble ni quede a la vez en entrenamiento y en prueba.

### 3.3 Variables redundantes o sin valor predictivo

In [ ]:
# Educacion vs NumEducacion: ¿cada nivel de texto corresponde a un único número?
df02.groupBy("Educacion", "NumEducacion").count().orderBy("NumEducacion").show(20)
print("Niveles de Educacion:", df02.select("Educacion").distinct().count(),
      "| combinaciones Educacion-NumEducacion:", df02.select("Educacion", "NumEducacion").distinct().count())

In [ ]:
# PesoFinal: ¿cambia según el ingreso?
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=pdNum, x="Ingreso", y="PesoFinal", order=["<=50K", ">50K"], ax=ejes[0])
ejes[0].set_title("PesoFinal según ingreso")
sns.histplot(data=pdNum, x="PesoFinal", hue="Ingreso", stat="density", common_norm=False, bins=50,
             hue_order=["<=50K", ">50K"], ax=ejes[1])
ejes[1].set_title("Distribución de PesoFinal")
plt.tight_layout()
plt.show()
print(pdNum.groupby("Ingreso")["PesoFinal"].median())
print("Correlación PesoFinal - Etiqueta:", round(pdNum["PesoFinal"].corr(pdNum["Etiqueta"]), 4))

**Lectura.** 
- **`Educacion` y `NumEducacion`:** hay 16 niveles de texto y exactamente 16 combinaciones, es decir, una relación 1 a 1. Son la misma información. Se conserva `NumEducacion` porque mantiene el orden de los niveles.
- **`PesoFinal`:** la mediana es casi igual en los dos grupos (178 811 frente a 176 729), las distribuciones se superponen por completo y la correlación con la etiqueta es −0.006. Además, por definición es un peso de muestreo del censo, no una característica de la persona.

**Decisión:** eliminar las dos columnas.

### 3.4 Categorías poco representativas

In [ ]:
# Categorías con menos de 100 registros (sin contar país, que se analiza aparte)
filasRaras = []
for c in ["ClaseTrabajo", "EstadoCivil", "Ocupacion", "Relacion", "Raza", "Sexo"]:
    for r in df02.groupBy(c).count().where("count < 100").collect():
        filasRaras.append({"Variable": c, "Categoria": r[c], "Registros": r["count"]})
pdRaras = pd.DataFrame(filasRaras)
print(pdRaras)

# Concentración del país de origen
pdPais = df02.groupBy("PaisOrigen").count().orderBy(F.desc("count")).toPandas()
pdPais["PaisOrigen"] = pdPais["PaisOrigen"].fillna("(faltante)")
top = pdPais.head(8).copy()
top.loc[len(top)] = ["Otros " + str(len(pdPais) - 8) + " países", pdPais["count"].iloc[8:].sum()]

fig, ejes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.barplot(data=pdRaras, x="Registros", y="Categoria", hue="Variable", dodge=False, ax=ejes[0])
ejes[0].set_title("Categorías con menos de 100 registros")
sns.barplot(data=top, x="count", y="PaisOrigen", color="steelblue", ax=ejes[1])
ejes[1].set_xscale("log")
ejes[1].set_title("Registros por país de origen (escala logarítmica)")
plt.tight_layout()
plt.show()
print(f"Estados Unidos concentra el {round(pdPais.iloc[0]['count']/totalReg*100, 1)} % de los registros")

**Lectura.** Cuatro categorías tienen menos de 100 registros: `Without-pay` (21), `Never-worked` (10), `Married-AF-spouse` (37) y `Armed-Forces` (15). En el país de origen, Estados Unidos concentra el 89.7 % y los otros 40 países tienen pocos registros cada uno (México, el segundo, tiene 951). Con tan pocos casos, un modelo no aprende nada estable de esas categorías y puede sobreajustarse a ellas. Además, los modelos de árboles de Spark exigen que `maxBins` sea mayor o igual al número de categorías, y 42 países obligarían a subirlo.

**Decisión:**
- `Without-pay` y `Never-worked` se unen en `Sin-pago`: las dos significan que no hay salario.
- `Married-AF-spouse` se une a `Married-civ-spouse`: los dos son "casado y viviendo con el cónyuge".
- El país se agrupa en `United-States`, `Otro` y `Desconocido`.
- `Armed-Forces` se deja como está, porque no tiene una ocupación parecida con la cual unirla.

### 3.5 Valores atípicos y asimetría

In [ ]:
# Regla IQR: atípico si está por fuera de [Q1 - 1.5·IQR, Q3 + 1.5·IQR]. Asimetría con F.skewness
filasAtip = []
for c in ["Edad", "HorasSemana", "GananciaCapital", "PerdidaCapital"]:
    q1, q3 = df02.approxQuantile(c, [0.25, 0.75], 0.0)
    iqr = q3 - q1
    limInf, limSup = q1 - 1.5*iqr, q3 + 1.5*iqr
    nAtip = df02.where((F.col(c) < limInf) | (F.col(c) > limSup)).count()
    asim = df02.select(F.skewness(c)).first()[0]
    filasAtip.append({"Variable": c, "Q1": q1, "Q3": q3, "Lim_inf": limInf, "Lim_sup": limSup,
                      "Atipicos": nAtip, "Pct": round(nAtip/totalReg*100, 2), "Asimetria": round(asim, 2)})
pdAtip = pd.DataFrame(filasAtip)
print(pdAtip)

fig, ejes = plt.subplots(1, 4, figsize=(20, 4.5))
for eje, c in zip(ejes, ["Edad", "HorasSemana", "GananciaCapital", "PerdidaCapital"]):
    sns.boxplot(data=pdNum, x="Ingreso", y=c, order=["<=50K", ">50K"], ax=eje)
    eje.set_title(c)
plt.tight_layout()
plt.show()

**Lectura.** 
- **Edad:** hay 216 registros (0.44 %) por encima de 78 años. Son edades posibles (máximo 90), así que **se conservan**.
- **HorasSemana:** la regla IQR marca el 27.6 %, porque la mitad central está entre 40 y 45 horas (la mayoría trabaja exactamente 40). Aquí la regla no sirve: trabajar 10 o 70 horas a la semana es perfectamente posible. **Se conservan**.
- **GananciaCapital y PerdidaCapital:** como Q1 = Q3 = 0, cualquier valor distinto de cero queda marcado como atípico (8.3 % y 4.7 %). Pero esos son justamente los registros informativos: el 61.7 % de quienes tienen ganancias gana >50K. Tampoco se eliminan. El problema real es la **asimetría** (11.89 y 4.57; una distribución simétrica tiene 0): pocos valores enormes y muchísimos ceros. **Decisión:** aplicar `log(1 + x)` en la sección 4.
- **El tope 99 999:** los 244 registros con ese valor son un código del censo para ganancias muy altas, así que se conservan como tales.

### 3.6 Resumen: problema → evidencia → decisión
| Problema | Evidencia | Decisión | Dónde |
|---|---|---|---|
| Valores faltantes | 3620 registros (7.4 %); quienes no tienen ocupación ganan >50K solo el 9.4 % | Reemplazar por `Desconocido` | 4.3 |
| Duplicados | 52 registros idénticos en todas las columnas | Eliminar | 4.1 |
| Variable redundante | `Educacion` ↔ `NumEducacion` en relación 1 a 1 | Eliminar `Educacion` | 4.2 |
| Variable sin valor predictivo | `PesoFinal`: correlación −0.006, es un peso muestral | Eliminar | 4.2 |
| Categorías raras | 4 categorías con menos de 100 registros; 41 países, 89.7 % EE. UU. | Agrupar | 4.4 |
| Asimetría extrema | Capital: asimetría 11.9 y 4.6; más del 90 % en cero | `log(1 + x)` | 4.5 |
| Atípicos en edad y horas | Valores posibles (edades hasta 90, 1 a 99 horas) | Conservar | — |
| Escalas distintas / categóricas en texto | Edad hasta 90, horas hasta 99, binarias en 0/1; categorías como texto | `StringIndexer`, `OneHotEncoder`, escaladores | 4.7 |
| Desbalance de clases | 23.9 % positivos | Métricas adecuadas y ajuste del umbral | 5.3, 6 |

# **4. Tratamiento de datos y pipeline**
Flujo completo, desde los archivos en HDFS hasta los vectores que reciben los modelos:

```
HDFS: adult.data + adult.test
   │  leerAdult() con esquema explícito                         → df00, df01
   │  unionByName + limpieza de etiquetas + Etiqueta 0/1         → df02  (48 842)
   ▼
TRATAMIENTO (sección 4.1 – 4.5)
   │  dropDuplicates → drop(PesoFinal, Educacion) → fillna("Desconocido")
   │  agrupación de categorías raras → log1p(capital)            → df03, df04
   ▼
DIVISIÓN (4.6)
   │  checkpoint en HDFS + randomSplit 80/20 (seed 1055)         → train / test
   ▼
PIPELINE DE CARACTERÍSTICAS (4.7) — fit solo con train
   │  StringIndexer ×7 → OneHotEncoder
   │  VectorAssembler → CaractArboles   (árboles)
   │  VectorAssembler → CaractOHE → StandardScaler → CaractEscaladas  (lineales, red neuronal)
   │                              → MinMaxScaler   → CaractNB         (Naive Bayes)
   ▼
MODELOS MLlib (sección 5) → evaluación (sección 6) → modelo final guardado en HDFS (5.5)
```

### 4.1 Eliminación de duplicados

In [ ]:
df03 = df02.dropDuplicates()
print(f"Antes: {totalReg}  Después: {df03.count()}  Eliminados: {totalReg - df03.count()}")

### 4.2 Eliminación de columnas

In [ ]:
# PesoFinal: peso muestral, no describe a la persona. Educacion: misma información que NumEducacion
df03 = df03.drop("PesoFinal", "Educacion")
print(df03.columns)

### 4.3 Valores faltantes → `Desconocido`

In [ ]:
# fillna(valor, subset): reemplaza los nulos de esas columnas por el texto indicado
df03 = df03.fillna("Desconocido", subset=["ClaseTrabajo", "Ocupacion", "PaisOrigen"])
# Verificación: ya no debe haber nulos
df03.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df03.columns]).show()

### 4.4 Agrupación de categorías poco representativas

In [ ]:
df03 = (df03
    .withColumn("ClaseTrabajo",
                F.when(F.col("ClaseTrabajo").isin("Without-pay", "Never-worked"), "Sin-pago")
                 .otherwise(F.col("ClaseTrabajo")))
    .withColumn("EstadoCivil",
                F.when(F.col("EstadoCivil") == "Married-AF-spouse", "Married-civ-spouse")
                 .otherwise(F.col("EstadoCivil")))
    .withColumn("PaisOrigen",
                F.when(F.col("PaisOrigen").isin("United-States", "Desconocido"), F.col("PaisOrigen"))
                 .otherwise("Otro")))

for c in ["ClaseTrabajo", "EstadoCivil", "PaisOrigen"]:
    resumenCategoria(df03, c)
print("Categorías por variable después de agrupar:")
for c in ["ClaseTrabajo", "EstadoCivil", "Ocupacion", "Relacion", "Raza", "Sexo", "PaisOrigen"]:
    print(f"  {c}: {df03.select(c).distinct().count()}")

### 4.5 Transformación logarítmica de las variables de capital
`log(1 + x)` comprime la escala: el 0 sigue siendo 0 y 99 999 pasa a 11.5. A los árboles no les afecta (solo usan el orden de los valores), pero a los modelos lineales y a la red neuronal les evita que unos pocos valores enormes dominen el ajuste.

In [ ]:
df04 = (df03.withColumn("GananciaCapital", F.log1p("GananciaCapital"))
            .withColumn("PerdidaCapital", F.log1p("PerdidaCapital"))
            .drop("Ingreso"))

# Antes y después, solo para quienes tienen ganancia de capital (el 92 % vale 0)
antes = df03.where("GananciaCapital > 0").select("GananciaCapital").toPandas()
despues = df04.where("GananciaCapital > 0").select("GananciaCapital").toPandas()
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(antes["GananciaCapital"], bins=40, ax=ejes[0], color="indianred")
ejes[0].set_title("GananciaCapital > 0, antes (dólares)")
sns.histplot(despues["GananciaCapital"], bins=40, ax=ejes[1], color="seagreen")
ejes[1].set_title("GananciaCapital > 0, después de log(1 + x)")
plt.tight_layout()
plt.show()
df04.select(F.round(F.skewness("GananciaCapital"), 2).alias("Asimetria_GananciaCapital"),
            F.round(F.skewness("PerdidaCapital"), 2).alias("Asimetria_PerdidaCapital")).show()

**Lectura.** La asimetría de `GananciaCapital` baja de 11.89 a 3.11, y en el histograma los valores dejan de amontonarse cerca de cero con un punto aislado en 99 999: ahora se reparten entre 5 y 11.5. En `PerdidaCapital` la asimetría casi no cambia (4.57 → 4.30), porque el 95 % de los valores son 0 y ninguna transformación puede repartir esa masa. Aun así, el rango se comprime de 0–4356 a 0–8.4, que es lo que importa para los modelos lineales.

### 4.6 División entrenamiento / prueba (80 / 20)
Antes de dividir, el DataFrame se **materializa** con `checkpoint()`, que lo escribe en HDFS (`Salario/checkpoints`). `dropDuplicates` reparte los datos entre particiones y el orden de las filas puede cambiar cada vez que Spark recalcula el DataFrame. Como `randomSplit` vuelve a ejecutarse en cada acción, sin este paso una misma fila podría caer unas veces en entrenamiento y otras en prueba. Se comprobó en las pruebas: sin el checkpoint, la matriz de confusión sumaba más filas que el conjunto de prueba. La semilla `seed=1055` hace que la división se pueda repetir.

In [ ]:
# checkpoint(): calcula df04 una vez, lo guarda en HDFS y corta el plan de ejecución.
# No se usa localCheckpoint() porque guarda los datos en la memoria de los ejecutores: si el clúster
# libera un ejecutor inactivo (asignación dinámica), esos bloques se pierden y los entrenamientos largos fallan.
sparkContextoLuis.setCheckpointDir("hdfs://10.195.34.34:9000/Formatos1055/BuenoLuis/Salario/checkpoints")
df04 = df04.checkpoint()

train, test = df04.randomSplit([0.8, 0.2], seed=1055)
train = train.cache()
test = test.cache()

filasSplit = []
for nombre, conjunto in [("Total", df04), ("Entrenamiento", train), ("Prueba", test)]:
    total = conjunto.count()
    positivos = conjunto.where("Etiqueta = 1").count()
    filasSplit.append({"Conjunto": nombre, "Registros": total, "Con_mayor50K": positivos,
                       "Pct_mayor50K": round(positivos/total*100, 2)})
pdSplit = pd.DataFrame(filasSplit)
print(pdSplit)
plt.figure(figsize=(6, 3.5))
sns.barplot(data=pdSplit, x="Conjunto", y="Pct_mayor50K", color="steelblue")
plt.title("% con >50K en cada conjunto")
plt.ylim(0, 30)
plt.show()

**Lectura.** Entrenamiento queda con 39 238 registros y prueba con 9552. La proporción de >50K es prácticamente la misma en el total (23.94 %), en entrenamiento (23.95 %) y en prueba (23.89 %), así que no hace falta un muestreo estratificado.

### 4.7 Pipeline de características
Cada modelo necesita las variables en un formato distinto, así que se arma un `Pipeline` que produce varias versiones del vector de características:

| Etapa | Qué hace | Para qué modelos |
|---|---|---|
| `StringIndexer` | Convierte cada categoría de texto en un número (0 = la más frecuente) | Todos |
| `OneHotEncoder` | Convierte cada índice en un vector de 0 y 1 (una posición por categoría) | Lineales, red neuronal y Naive Bayes |
| `VectorAssembler` → `CaractArboles` | 5 numéricas + 7 índices de categoría (12 posiciones) | Árbol, Random Forest, GBT |
| `VectorAssembler` → `CaractOHE` | 5 numéricas + categorías en one-hot | Base de las dos siguientes |
| `StandardScaler` → `CaractEscaladas` | Media 0 y desviación 1 en cada posición | Regresión logística, Linear SVC, perceptrón |
| `MinMaxScaler` → `CaractNB` | Lleva todo al rango [0, 1] | Naive Bayes (no admite valores negativos) |

**¿Por qué dos codificaciones?** Para los modelos lineales, el índice de `StringIndexer` inventa un orden que no existe: diría que `Sales = 4` es "más" que `Craft-repair = 1`. El one-hot evita ese problema. Los árboles, en cambio, reconocen los índices como categorías gracias a los metadatos que deja `StringIndexer`, y trabajan mejor con menos columnas.

El pipeline se ajusta (`fit`) **solo con entrenamiento**: el diccionario de categorías, las medias, las desviaciones y los mínimos y máximos se aprenden de esos datos y después se aplican igual a prueba. Así no se filtra información del conjunto de prueba. `handleInvalid="keep"` evita errores si en prueba aparece una categoría que no estaba en entrenamiento.

In [ ]:
categoricas = ["ClaseTrabajo", "EstadoCivil", "Ocupacion", "Relacion", "Raza", "Sexo", "PaisOrigen"]
numericas = ["Edad", "NumEducacion", "GananciaCapital", "PerdidaCapital", "HorasSemana"]

etapas = [StringIndexer(inputCol=c, outputCol=c+"_IND", handleInvalid="keep") for c in categoricas]
etapas += [OneHotEncoder(inputCols=[c+"_IND" for c in categoricas], outputCols=[c+"_OHE" for c in categoricas])]
etapas += [
    VectorAssembler(inputCols=numericas + [c+"_IND" for c in categoricas], outputCol="CaractArboles"),
    VectorAssembler(inputCols=numericas + [c+"_OHE" for c in categoricas], outputCol="CaractOHE"),
    StandardScaler(inputCol="CaractOHE", outputCol="CaractEscaladas", withMean=True, withStd=True),
    MinMaxScaler(inputCol="CaractOHE", outputCol="CaractNB"),
]

pipelinePrep = Pipeline(stages=etapas)
modeloPrep = pipelinePrep.fit(train)          # aprende solo de entrenamiento
trainPrep = modeloPrep.transform(train).cache()
testPrep = modeloPrep.transform(test).cache()

tamOHE = len(trainPrep.first()["CaractEscaladas"])
print(f"Etapas del pipeline: {len(modeloPrep.stages)}")
print(f"Posiciones en CaractArboles: {len(trainPrep.first()['CaractArboles'])}")
print(f"Posiciones en CaractOHE / CaractEscaladas / CaractNB: {tamOHE}")

In [ ]:
# Recorrido de un registro por las etapas del pipeline
registro = trainPrep.where("Etiqueta = 1").limit(1)
registro.select(numericas + categoricas).show(truncate=False)
registro.select([c+"_IND" for c in categoricas]).show(truncate=False)
registro.select("Ocupacion", "Ocupacion_IND", "Ocupacion_OHE", "Sexo", "Sexo_IND", "Sexo_OHE").show(truncate=False)
registro.select("CaractArboles").show(truncate=False)
registro.select("CaractEscaladas").show(truncate=False)

**Lectura.** El pipeline tiene 12 etapas: 7 `StringIndexer`, 1 `OneHotEncoder`, 2 `VectorAssembler` y 2 escaladores. El registro de ejemplo es una mujer de 19 años, nunca casada, que vive con sus padres (`Own-child`) y trabaja 20 horas en una ocupación profesional, con una ganancia de capital de unos 14 000 dólares (log = 9.55); gana >50K. Así se transforma en cada etapa:
- **`StringIndexer`:** `Prof-specialty` → 0, porque es la ocupación más frecuente en entrenamiento.
- **`OneHotEncoder`:** ese 0 pasa a `(15,[0],[1.0])`, un vector de 15 posiciones con un 1 en la primera. Son 15 categorías más el lugar reservado por `handleInvalid="keep"`, menos una que queda implícita (`dropLast`).
- **`CaractArboles`:** el registro queda en 12 números.
- **`CaractEscaladas`:** queda en 50, ya estandarizados. Por ejemplo, la edad queda en −1.43 (muy por debajo del promedio) y la ganancia de capital en +3.6 desviaciones.

### 4.8 Resumen del flujo de DataFrames
| DataFrame | Qué contiene | Registros |
|---|---|---|
| `df00`, `df01` | `adult.data` y `adult.test` leídos desde HDFS con el esquema | 32 561 y 16 281 |
| `df02` | Unión + etiquetas limpias + `Etiqueta` | 48 842 |
| `df03` | Sin duplicados, sin `PesoFinal`/`Educacion`, faltantes como `Desconocido`, categorías agrupadas | 48 790 |
| `df04` | `df03` + logaritmo de capital, materializado con `checkpoint` en HDFS | 48 790 |
| `train` / `test` | División 80/20 con semilla 1055 | 39 229 / 9561 |
| `trainPrep` / `testPrep` | `train` / `test` + columnas del pipeline (`_IND`, `_OHE` y los 4 vectores) | 39 229 / 9561 |

# **5. Aplicación de modelos MLlib**
### 5.1 Modelos base
Se entrenan siete clasificadores de `pyspark.ml.classification` con parámetros razonables de partida:

| Modelo | Idea | Vector | Parámetros |
|---|---|---|---|
| Regresión logística | Combinación lineal de las variables pasada por una sigmoide → probabilidad | `CaractEscaladas` | 100 iteraciones |
| Árbol de decisión | Reglas "si-entonces" que parten los datos por la variable más informativa | `CaractArboles` | profundidad 8 |
| Random Forest | 100 árboles entrenados sobre muestras distintas; vota la mayoría | `CaractArboles` | 100 árboles, profundidad 10 |
| Gradient-Boosted Trees (GBT) | Árboles en secuencia; cada uno corrige los errores de los anteriores | `CaractArboles` | 50 árboles, profundidad 5 |
| Naive Bayes | Teorema de Bayes, suponiendo variables independientes | `CaractNB` | multinomial |
| Linear SVC | Hiperplano que separa las clases con el mayor margen | `CaractEscaladas` | 100 iteraciones |
| Perceptrón multicapa | Red neuronal: entrada → 16 → 8 neuronas → 2 salidas | `CaractEscaladas` | 100 iteraciones |

Se fija la semilla en los modelos que tienen componente aleatoria y se mide el tiempo de entrenamiento de cada uno.

In [ ]:
modelos = {
    "Regresión logística":  LogisticRegression(featuresCol="CaractEscaladas", labelCol="Etiqueta", maxIter=100),
    "Árbol de decisión":    DecisionTreeClassifier(featuresCol="CaractArboles", labelCol="Etiqueta", maxDepth=8, seed=1055),
    "Random Forest":        RandomForestClassifier(featuresCol="CaractArboles", labelCol="Etiqueta",
                                                   numTrees=100, maxDepth=10, seed=1055),
    "GBT":                  GBTClassifier(featuresCol="CaractArboles", labelCol="Etiqueta", maxIter=50, maxDepth=5, seed=1055),
    "Naive Bayes":          NaiveBayes(featuresCol="CaractNB", labelCol="Etiqueta"),
    "Linear SVC":           LinearSVC(featuresCol="CaractEscaladas", labelCol="Etiqueta", maxIter=100),
    # capas: [entrada = tamaño del vector, oculta 1, oculta 2, salida = 2 clases]
    "Perceptrón multicapa": MultilayerPerceptronClassifier(featuresCol="CaractEscaladas", labelCol="Etiqueta",
                                                           layers=[tamOHE, 16, 8, 2], maxIter=100, seed=1055),
}

modelosEntrenados, predicciones, tiempos = {}, {}, {}
for nombre, modelo in modelos.items():
    inicio = time.time()
    modelosEntrenados[nombre] = modelo.fit(trainPrep)                           # entrenamiento
    tiempos[nombre] = time.time() - inicio
    predicciones[nombre] = modelosEntrenados[nombre].transform(testPrep).cache()  # predicción sobre prueba
    print(f"{nombre}: entrenado en {round(tiempos[nombre], 1)} s")

In [ ]:
# Así se ve una predicción: probabilidad de cada clase y la clase elegida (umbral 0.5)
predicciones["GBT"].select("Etiqueta", "probability", "prediction").show(5, truncate=False)

In [ ]:
evalROC = BinaryClassificationEvaluator(labelCol="Etiqueta", metricName="areaUnderROC")
evalPR = BinaryClassificationEvaluator(labelCol="Etiqueta", metricName="areaUnderPR")

def evaluar(nombre, pred, tiempo=None):
    # Para el AUC se usa la probabilidad. Linear SVC no la tiene y se usa rawPrediction (distancia al hiperplano).
    # En Naive Bayes rawPrediction es una log-probabilidad sin normalizar y daría un AUC incorrecto.
    columna = "probability" if "probability" in pred.columns else "rawPrediction"
    evalROC.setRawPredictionCol(columna)
    evalPR.setRawPredictionCol(columna)

    # Matriz de confusión: conteo por (real, predicho)
    m = {(r["Etiqueta"], int(r["prediction"])): r["count"]
         for r in pred.groupBy("Etiqueta", "prediction").count().collect()}
    vp, fn = m.get((1, 1), 0), m.get((1, 0), 0)
    fp, vn = m.get((0, 1), 0), m.get((0, 0), 0)
    precision = vp/(vp+fp) if (vp+fp) > 0 else 0.0
    recall = vp/(vp+fn) if (vp+fn) > 0 else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"Modelo": nombre,
            "AUC_ROC": round(evalROC.evaluate(pred), 4),
            "AUC_PR": round(evalPR.evaluate(pred), 4),
            "Accuracy": round((vp+vn)/(vp+vn+fp+fn), 4),
            "Precision": round(precision, 3),
            "Recall": round(recall, 3),
            "F1": round(f1, 3),
            "VP": vp, "FN": fn, "FP": fp, "VN": vn,
            "Tiempo_s": round(tiempos.get(nombre, 0) if tiempo is None else tiempo, 1)}

pdBase = pd.DataFrame([evaluar(n, p) for n, p in predicciones.items()]).sort_values("AUC_ROC", ascending=False)
pdBase

**Lectura.** 
- **GBT** es el mejor de los modelos base en AUC-ROC (0.917), AUC-PR (0.788), accuracy (85.9 %) y F1 (0.681).
- **Random Forest** (0.913) y el **perceptrón multicapa** (0.904) le siguen.
- Los modelos lineales, **regresión logística** (0.901) y **Linear SVC** (0.898), quedan muy parejos entre sí.
- El **árbol de decisión** tiene el AUC más bajo de los modelos de árboles (0.888), pero el segundo mejor F1 (0.669), porque detecta más personas con >50K que los demás con el umbral de 0.5.
- **Naive Bayes** (AUC 0.866, accuracy 76.6 %) apenas supera la línea base.

Con el umbral por defecto, todos menos Naive Bayes encuentran solo entre el 58 % y el 64 % de las personas con >50K. Eso motiva el ajuste de la sección siguiente.

### 5.2 Ajuste de hiperparámetros con validación cruzada
Se ajustan los dos modelos más prometedores de familias distintas: **GBT** (el mejor de los modelos base) y la **regresión logística** (el mejor modelo lineal y el más fácil de interpretar).

Para no usar el conjunto de prueba en ninguna decisión, el entrenamiento se divide otra vez:
- **Ajuste (85 % de `train`)**: aquí corre `CrossValidator` con 3 pliegues. Cada combinación de parámetros se entrena 3 veces, cada vez dejando por fuera un pliegue distinto para medir el AUC.
- **Validación (15 % de `train`)**: se usa solo para elegir el umbral de decisión (5.3).
- **Prueba (`test`)**: se usa solo al final, para medir el desempeño de los modelos ya decididos.

`parallelism=2` entrena dos modelos a la vez en el clúster.

In [ ]:
ajustePrep, validPrep = trainPrep.randomSplit([0.85, 0.15], seed=1055)
ajustePrep = ajustePrep.cache()
validPrep = validPrep.cache()
print(f"Ajuste: {ajustePrep.count()}  |  Validación: {validPrep.count()}")

evalAUC = BinaryClassificationEvaluator(labelCol="Etiqueta", metricName="areaUnderROC")

def tablaCV(grid, modeloCV):
    filas = [{**{p.name: v for p, v in pm.items()}, "AUC_promedio_CV": round(m, 4)}
             for pm, m in zip(grid, modeloCV.avgMetrics)]
    return pd.DataFrame(filas).sort_values("AUC_promedio_CV", ascending=False).reset_index(drop=True)

In [ ]:
# GBT: profundidad de los árboles y número de árboles
gbtCV = GBTClassifier(featuresCol="CaractArboles", labelCol="Etiqueta", seed=1055)
gridGBT = (ParamGridBuilder()
           .addGrid(gbtCV.maxDepth, [4, 6])
           .addGrid(gbtCV.maxIter, [50, 100])
           .build())
cvGBT = CrossValidator(estimator=gbtCV, estimatorParamMaps=gridGBT, evaluator=evalAUC,
                       numFolds=3, parallelism=2, seed=1055)
inicio = time.time()
modeloCVGBT = cvGBT.fit(ajustePrep)
print(f"Validación cruzada GBT: {len(gridGBT)} combinaciones × 3 pliegues en {round(time.time() - inicio, 1)} s")
tablaCV(gridGBT, modeloCVGBT)

In [ ]:
# Regresión logística: regularización (regParam) y mezcla L1/L2 (elasticNetParam: 0 = L2, 0.5 = mitad y mitad)
lrCV = LogisticRegression(featuresCol="CaractEscaladas", labelCol="Etiqueta", maxIter=100)
gridLR = (ParamGridBuilder()
          .addGrid(lrCV.regParam, [0.0, 0.01, 0.1])
          .addGrid(lrCV.elasticNetParam, [0.0, 0.5])
          .build())
cvLR = CrossValidator(estimator=lrCV, estimatorParamMaps=gridLR, evaluator=evalAUC,
                      numFolds=3, parallelism=2, seed=1055)
inicio = time.time()
modeloCVLR = cvLR.fit(ajustePrep)
print(f"Validación cruzada regresión logística: {len(gridLR)} combinaciones × 3 pliegues en {round(time.time() - inicio, 1)} s")
tablaCV(gridLR, modeloCVLR)

**Lectura.** 
- **GBT:** la mejor combinación es profundidad 4 con 100 árboles (AUC promedio 0.9155). Los árboles de profundidad 6 rinden peor que los de 4, con 50 o con 100 árboles: los árboles más profundos se sobreajustan. Con árboles poco profundos, en cambio, agregar más árboles sí ayuda (0.9136 → 0.9155). Como el mejor valor quedó en el borde de la grilla (100), vale la pena probar más árboles en un trabajo futuro.
- **Regresión logística:** la mejor opción es **sin regularización** (`regParam = 0`, AUC 0.9002), que coincide con los parámetros por defecto. Regularizar empeora el modelo, sobre todo con la mezcla L1 (0.878 con `regParam = 0.1` y `elasticNetParam = 0.5`), porque L1 anula variables que sí aportan. Con 33 mil registros y solo 50 variables, el modelo lineal no corre riesgo de sobreajuste. Por eso se conserva el modelo base.
- **Costo:** la validación cruzada de GBT es la parte más costosa del cuaderno, porque entrena 12 modelos más el modelo final. La de la regresión logística cuesta una décima parte de ese tiempo.

### 5.3 Umbral de decisión
Por defecto, el modelo predice `>50K` cuando la probabilidad supera 0.5. Con solo 24 % de positivos, ese umbral deja escapar a muchas personas con >50K. Se prueba una serie de umbrales sobre el conjunto de **validación** con el mejor GBT de la validación cruzada, y se elige el que maximiza el F1.

In [ ]:
mejorGBTcv = modeloCVGBT.bestModel
pdValid = (mejorGBTcv.transform(validPrep)
           .select(vector_to_array("probability")[1].alias("Prob"), "Etiqueta").toPandas())

filasUmbral = []
for u in np.round(np.arange(0.20, 0.71, 0.05), 2):
    pred = (pdValid["Prob"] > u).astype(int)
    vp = int(((pred == 1) & (pdValid["Etiqueta"] == 1)).sum())
    fp = int(((pred == 1) & (pdValid["Etiqueta"] == 0)).sum())
    fn = int(((pred == 0) & (pdValid["Etiqueta"] == 1)).sum())
    p = vp/(vp+fp) if vp+fp else 0
    r = vp/(vp+fn) if vp+fn else 0
    filasUmbral.append({"Umbral": u, "Precision": round(p, 3), "Recall": round(r, 3),
                        "F1": round(2*p*r/(p+r), 3) if p+r else 0, "Marcados_>50K": vp+fp})
pdUmbral = pd.DataFrame(filasUmbral)
umbral = float(pdUmbral.loc[pdUmbral["F1"].idxmax(), "Umbral"])
print(pdUmbral)
print(f"Umbral elegido (máximo F1 en validación): {umbral}")

plt.figure(figsize=(9, 4.5))
for m, color in [("Precision", "tab:red"), ("Recall", "tab:purple"), ("F1", "tab:brown")]:
    plt.plot(pdUmbral["Umbral"], pdUmbral[m], marker="o", label=m, color=color)
plt.axvline(umbral, color="black", linestyle="--", label=f"Umbral elegido = {umbral}")
plt.axvline(0.5, color="gray", linestyle=":", label="Umbral por defecto = 0.5")
plt.title("Precisión, recall y F1 según el umbral (conjunto de validación, GBT ajustado)")
plt.xlabel("Umbral de probabilidad")
plt.legend()
plt.show()

**Lectura.** Al bajar el umbral, el modelo marca a más personas como >50K: sube el recall y baja la precisión. El F1 tiene su máximo en **0.30** (0.698), frente a 0.665 con el umbral por defecto. Con 0.30, en validación, el recall pasa de 0.605 a 0.796 y la precisión baja de 0.737 a 0.621. El F1 se mantiene por encima de 0.68 entre 0.25 y 0.40, así que la elección no depende de un punto exacto. Esta tabla también sirve para elegir otro umbral según el uso (sección 7).

### 5.4 Modelos finales
Con los mejores parámetros y el umbral elegido, el modelo se vuelve a entrenar con **todo** el conjunto de entrenamiento (ajuste + validación), para aprovechar todos los datos. El umbral queda dentro del modelo con `setThresholds`: Spark elige la clase con mayor `probabilidad / threshold`, así que `thresholds = [1 - u, u]` equivale a predecir `>50K` cuando la probabilidad supera `u`.

In [ ]:
mejorLRcv = modeloCVLR.bestModel
print(f"GBT: maxDepth = {mejorGBTcv.getMaxDepth()}, maxIter = {mejorGBTcv.getMaxIter()}, umbral = {umbral}")
print(f"Regresión logística: regParam = {mejorLRcv.getRegParam()}, elasticNetParam = {mejorLRcv.getElasticNetParam()}")

finales = {
    "GBT ajustado": GBTClassifier(featuresCol="CaractArboles", labelCol="Etiqueta", seed=1055,
                                  maxDepth=mejorGBTcv.getMaxDepth(), maxIter=mejorGBTcv.getMaxIter()
                                  ).setThresholds([1 - umbral, umbral]),
}
# Si la validación cruzada eligió los mismos parámetros del modelo base (regParam = 0), reentrenarlo daría
# un modelo idéntico: se conserva el base. Si eligió otros, se entrena la versión ajustada.
if mejorLRcv.getRegParam() > 0:
    finales["Regresión logística ajustada"] = LogisticRegression(featuresCol="CaractEscaladas", labelCol="Etiqueta", maxIter=100,
                                                                 regParam=mejorLRcv.getRegParam(),
                                                                 elasticNetParam=mejorLRcv.getElasticNetParam())
else:
    print("Regresión logística: la validación cruzada eligió los parámetros por defecto; se conserva el modelo base.")

for nombre, modelo in finales.items():
    inicio = time.time()
    modelosEntrenados[nombre] = modelo.fit(trainPrep)
    tiempos[nombre] = time.time() - inicio
    predicciones[nombre] = modelosEntrenados[nombre].transform(testPrep).cache()
    print(f"{nombre}: entrenado en {round(tiempos[nombre], 1)} s")

### 5.5 Implantación: guardar el modelo final en HDFS
Se arma un `PipelineModel` con todas las etapas de preparación más el GBT ajustado, y se guarda en HDFS. Así, cualquier otro cuaderno o aplicación del clúster puede cargarlo y predecir directamente sobre datos **crudos** (con las columnas de `df04`), sin repetir la preparación a mano.

In [ ]:
modeloCompleto = PipelineModel(stages=modeloPrep.stages + [modelosEntrenados["GBT ajustado"]])
rutaModelo = "hdfs://10.195.34.34:9000/Formatos1055/BuenoLuis/Salario/modelo_gbt_salario"
modeloCompleto.write().overwrite().save(rutaModelo)
print("Modelo guardado en:", rutaModelo)

In [ ]:
!/mnt/sda1/Cluster/Hadoop/bin/hadoop fs -ls /Formatos1055/BuenoLuis/Salario/modelo_gbt_salario/stages

In [ ]:
# Se carga el modelo desde HDFS y se aplica sobre los datos de prueba SIN preparar
modeloCargado = PipelineModel.load(rutaModelo)
predCargado = modeloCargado.transform(test)
predCargado.select("Edad", "NumEducacion", "EstadoCivil", "Ocupacion", "HorasSemana",
                   "Etiqueta", "probability", "prediction").show(5, truncate=False)

# Verificación: las predicciones del modelo cargado deben ser idénticas a las del modelo en memoria
a = predCargado.select("prediction").toPandas()["prediction"].values
b = predicciones["GBT ajustado"].select("prediction").toPandas()["prediction"].values
print(f"Predicciones idénticas: {(a == b).all()}  ({len(a)} registros)")

**Lectura.** El modelo completo tiene 13 etapas: las 12 de preparación más el GBT ajustado, con su umbral incluido. Se guardó en HDFS, se volvió a cargar y se aplicó directamente sobre `test` **sin preparar**: las 9552 predicciones son idénticas a las del modelo en memoria. En el ejemplo, cinco jóvenes de 17 años sin ocupación registrada reciben una probabilidad de 1.2 % de ganar más de 50K.

# **6. Comparativa general de modelos**
Métricas sobre el conjunto de prueba (clase positiva = ingreso >50K):
- **AUC-ROC**: qué tan bien ordena el modelo a las personas de menor a mayor probabilidad de ganar >50K, sin depender del umbral. 0.5 = azar, 1 = perfecto.
- **AUC-PR**: área bajo la curva precisión-recall. Es más exigente cuando la clase positiva es minoritaria; un modelo al azar daría ~0.24 (la proporción de positivos).
- **Accuracy**: porcentaje de aciertos totales. Un modelo que diga siempre `<=50K` ya acierta el ~76 %, así que ese es el piso a superar.
- **Precisión**: de las personas que el modelo predice con >50K, cuántas lo son de verdad.
- **Recall**: de las personas que de verdad ganan >50K, cuántas encuentra el modelo.
- **F1**: equilibrio entre precisión y recall.
- **Tiempo** de entrenamiento.

Precisión, recall y F1 se calculan para la clase >50K desde la matriz de confusión, no con las versiones ponderadas (`weightedPrecision`), porque esas promedian las dos clases y la mayoritaria tapa lo que pasa con la minoritaria.

### 6.1 Tabla general y ranking

In [ ]:
pdResultados = pd.DataFrame([evaluar(n, p) for n, p in predicciones.items()])
# El ranking usa AUC-ROC, AUC-PR y F1. El accuracy se reporta, pero no entra: con 76 % de una clase
# premia al modelo que predice más "<=50K", y castiga a un modelo que movió su umbral a propósito para encontrar más ">50K".
for m in ["AUC_ROC", "AUC_PR", "F1"]:
    pdResultados["Pos_" + m] = pdResultados[m].rank(ascending=False, method="min").astype(int)
pdResultados["Posicion_promedio"] = pdResultados[[c for c in pdResultados.columns if c.startswith("Pos_")]].mean(axis=1).round(2)
pdResultados = pdResultados.sort_values(["Posicion_promedio", "AUC_ROC"], ascending=[True, False]).reset_index(drop=True)
lineaBase = test.where("Etiqueta = 0").count() / test.count()
print(f"Línea base (predecir siempre <=50K): accuracy = {round(lineaBase, 4)}")
pdResultados

### 6.2 Métricas y tiempo de entrenamiento

In [ ]:
pdMetricas = pdResultados.melt(id_vars="Modelo", value_vars=["AUC_ROC", "AUC_PR", "Accuracy", "Precision", "Recall", "F1"],
                               var_name="Métrica", value_name="Valor")
fig, ejes = plt.subplots(1, 2, figsize=(22, 6.5), gridspec_kw={"width_ratios": [3, 1]})
sns.barplot(data=pdMetricas, x="Modelo", y="Valor", hue="Métrica", ax=ejes[0])
ejes[0].axhline(lineaBase, color="black", linestyle=":", label="Línea base accuracy")
ejes[0].set_title("Métricas en el conjunto de prueba (clase positiva = >50K)")
ejes[0].set_ylim(0, 1)
ejes[0].set_xlabel("")
ejes[0].tick_params(axis="x", rotation=25)
ejes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=7)
sns.barplot(data=pdResultados, y="Modelo", x="Tiempo_s", color="steelblue", ax=ejes[1])
ejes[1].set_title("Tiempo de entrenamiento (s)")
ejes[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Costo vs. desempeño: ¿cuánto AUC se gana por cada segundo de entrenamiento?
plt.figure(figsize=(10, 6))
plt.scatter(pdResultados["Tiempo_s"], pdResultados["AUC_ROC"], s=120, color="steelblue")
for fila in pdResultados.itertuples():
    plt.annotate(fila.Modelo, (fila.Tiempo_s, fila.AUC_ROC), textcoords="offset points", xytext=(6, 4))
plt.xlabel("Tiempo de entrenamiento (s)")
plt.ylabel("AUC-ROC en prueba")
plt.title("Costo vs. desempeño")
plt.grid(alpha=0.3)
plt.show()

### 6.3 Curvas ROC y precisión-recall

In [ ]:
# Puntaje de la clase 1 de cada modelo (probabilidad, o distancia al hiperplano en Linear SVC)
puntajes = {}
for nombre in pdResultados["Modelo"]:
    pred = predicciones[nombre]
    columna = "probability" if "probability" in pred.columns else "rawPrediction"
    puntajes[nombre] = pred.select(vector_to_array(columna)[1].alias("Puntaje"), "Etiqueta").toPandas()

fig, ejes = plt.subplots(1, 2, figsize=(18, 7))
for nombre, pdp in puntajes.items():
    fpr, tpr, _ = roc_curve(pdp["Etiqueta"], pdp["Puntaje"])
    ejes[0].plot(fpr, tpr, label=f"{nombre} ({pdResultados.set_index('Modelo').loc[nombre, 'AUC_ROC']})")
    prec, rec, _ = precision_recall_curve(pdp["Etiqueta"], pdp["Puntaje"])
    ejes[1].plot(rec, prec, label=f"{nombre} ({pdResultados.set_index('Modelo').loc[nombre, 'AUC_PR']})")
ejes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Azar (0.5)")
ejes[0].set_title("Curvas ROC (AUC-ROC)")
ejes[0].set_xlabel("Tasa de falsos positivos")
ejes[0].set_ylabel("Tasa de verdaderos positivos (recall)")
ejes[1].axhline(1 - lineaBase, linestyle="--", color="gray", label=f"Azar ({round(1 - lineaBase, 2)})")
ejes[1].set_title("Curvas precisión-recall (AUC-PR)")
ejes[1].set_xlabel("Recall")
ejes[1].set_ylabel("Precisión")
for eje in ejes:
    eje.legend(loc="lower right" if eje is ejes[0] else "lower left", fontsize=9)
plt.tight_layout()
plt.show()

### 6.4 Matrices de confusión
Cada matriz cruza el valor real (filas) con el predicho (columnas):
- **Real 0 / Pred. 0**: gana `<=50K` y el modelo acierta (verdaderos negativos).
- **Real 0 / Pred. 1**: gana `<=50K` pero el modelo dice `>50K` (falsos positivos).
- **Real 1 / Pred. 0**: gana `>50K` pero el modelo dice `<=50K` (falsos negativos).
- **Real 1 / Pred. 1**: gana `>50K` y el modelo acierta (verdaderos positivos).

In [ ]:
fig, ejes = plt.subplots(3, 3, figsize=(18, 14))
for eje in ejes.flatten()[len(pdResultados):]:
    eje.axis("off")   # cuadros sobrantes de la cuadrícula
for eje, fila in zip(ejes.flatten(), pdResultados.itertuples()):
    matriz = [[fila.VN, fila.FP], [fila.FN, fila.VP]]
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=eje,
                xticklabels=["Pred. 0", "Pred. 1"], yticklabels=["Real 0", "Real 1"])
    eje.set_title(f"{fila.Modelo}\nAccuracy = {fila.Accuracy} | Recall = {fila.Recall}")
plt.suptitle("Matrices de confusión en el conjunto de prueba", fontsize=14)
plt.tight_layout()
plt.show()

**Lectura.** 
- **GBT ajustado** (umbral 0.30) encuentra a 1849 de las 2282 personas con >50K (81.0 %), frente a 1432 (62.8 %) del GBT con umbral 0.5. El costo son más falsos positivos: 1073 frente a 493.
- **Random Forest** es el más conservador de los buenos modelos: tiene la menor cantidad de falsos positivos (410), pero deja escapar a 965 personas con >50K.
- **Naive Bayes** encuentra muchas personas con >50K (1786), pero con 1735 falsos positivos: acierta apenas la mitad de las veces que dice `>50K`.
- Los demás modelos, con el umbral de 0.5, tienen más falsos negativos que falsos positivos: ante la duda, se inclinan por la clase mayoritaria.

### 6.5 ¿Las diferencias son reales? Intervalos de confianza con bootstrap
Con un solo conjunto de prueba, una diferencia de 0.003 en AUC podría deberse al azar de qué personas cayeron en prueba. El **bootstrap** remuestrea el conjunto de prueba 500 veces (con reemplazo), recalcula el AUC en cada muestra y toma el percentil 2.5 y 97.5 como intervalo de confianza del 95 %. Para comparar dos modelos se usa la **misma** muestra en los dos y se mira la diferencia: si su intervalo no incluye el 0, la diferencia es significativa.

In [ ]:
rng = np.random.default_rng(1055)
etiquetas = puntajes["GBT"]["Etiqueta"].values
assert all((pdp["Etiqueta"].values == etiquetas).all() for pdp in puntajes.values())   # mismo orden en todos
n, B = len(etiquetas), 500
muestras = [rng.integers(0, n, n) for _ in range(B)]

aucBoot = {nombre: np.array([roc_auc_score(etiquetas[idx], pdp["Puntaje"].values[idx]) for idx in muestras])
           for nombre, pdp in puntajes.items()}
pdIC = pd.DataFrame([{"Modelo": n_, "AUC": round(roc_auc_score(etiquetas, puntajes[n_]["Puntaje"]), 4),
                      "IC95_inf": round(np.percentile(v, 2.5), 4), "IC95_sup": round(np.percentile(v, 97.5), 4)}
                     for n_, v in aucBoot.items()]).sort_values("AUC", ascending=False).reset_index(drop=True)
print(pdIC)

# Diferencias pareadas frente al modelo final
comparaciones = [("GBT ajustado", "Random Forest"), ("GBT ajustado", "GBT"), ("GBT ajustado", "Regresión logística"),
                 ("Random Forest", "Perceptrón multicapa")]
pdDif = pd.DataFrame([{"Comparación": f"{a_} − {b_}", "Diferencia_AUC": round((aucBoot[a_] - aucBoot[b_]).mean(), 4),
                       "IC95_inf": round(np.percentile(aucBoot[a_] - aucBoot[b_], 2.5), 4),
                       "IC95_sup": round(np.percentile(aucBoot[a_] - aucBoot[b_], 97.5), 4)} for a_, b_ in comparaciones])
pdDif["Significativa"] = (pdDif["IC95_inf"] > 0) | (pdDif["IC95_sup"] < 0)
print(pdDif)

plt.figure(figsize=(9, 5))
plt.errorbar(pdIC["AUC"], pdIC["Modelo"], xerr=[pdIC["AUC"] - pdIC["IC95_inf"], pdIC["IC95_sup"] - pdIC["AUC"]],
             fmt="o", capsize=5, color="steelblue")
plt.gca().invert_yaxis()
plt.title("AUC-ROC en prueba con intervalo de confianza del 95 % (bootstrap, 500 muestras)")
plt.xlabel("AUC-ROC")
plt.grid(alpha=0.3)
plt.show()

**Lectura.** 
- **Los intervalos individuales se traslapan:** GBT ajustado [0.910, 0.923], GBT [0.910, 0.923] y Random Forest [0.906, 0.919].
- **La prueba pareada sí separa a GBT de Random Forest:** la diferencia es **0.0041 [0.0024, 0.0057]**. El intervalo no incluye el 0, así que la ventaja de GBT es real, aunque pequeña.
- **GBT ajustado frente a la regresión logística:** la diferencia es mayor, 0.0165 [0.0137, 0.0194], también significativa.
- **GBT ajustado frente al GBT base:** la diferencia es **0.0001 [−0.0009, 0.0011]**, que no es significativa. El ajuste de hiperparámetros casi no cambia la capacidad de ordenar a las personas. La mejora real del ajuste está en el umbral, que cambia el F1 y el recall (6.6).
- **Random Forest frente al perceptrón multicapa:** la diferencia es 0.0093 [0.0064, 0.0120], también significativa: la red neuronal no alcanza a Random Forest.

### 6.6 Antes y después del ajuste

In [ ]:
gbtAjustado = modelosEntrenados["GBT ajustado"]
predGBT05 = gbtAjustado.copy({gbtAjustado.thresholds: [0.5, 0.5]}).transform(testPrep)
pdAntesDespues = pd.DataFrame([
    evaluar("GBT base (umbral 0.5)", predicciones["GBT"], tiempos["GBT"]),
    evaluar("GBT ajustado (umbral 0.5)", predGBT05, tiempos["GBT ajustado"]),
    evaluar(f"GBT ajustado (umbral {umbral})", predicciones["GBT ajustado"], tiempos["GBT ajustado"]),
    evaluar("Regresión logística base", predicciones["Regresión logística"], tiempos["Regresión logística"]),
] + ([evaluar("Regresión logística ajustada", predicciones["Regresión logística ajustada"])]
     if "Regresión logística ajustada" in predicciones else []))
pdAntesDespues

**Lectura.** Con el umbral de 0.5, el GBT ajustado mejora muy poco al GBT base: AUC 0.9171 → 0.9173 y F1 0.681 → 0.684. El cambio grande viene del **umbral 0.30**:
- El F1 sube a **0.711**.
- El recall sube de 0.634 a **0.810**: encuentra 402 personas con >50K más que el mismo modelo con 0.5.
- A cambio, la precisión baja de 0.743 a 0.633 y el accuracy de 86.0 % a 84.2 %.

Es un intercambio buscado: el accuracy baja porque el modelo deja de "ir a la segura" con la clase mayoritaria. Para la regresión logística, la validación cruzada confirmó los parámetros por defecto, así que el modelo base es el definitivo.

### 6.7 Importancia de variables

In [ ]:
nombresArboles = numericas + categoricas
fig, ejes = plt.subplots(1, 2, figsize=(16, 5))
for eje, nombre in zip(ejes, ["GBT ajustado", "Random Forest"]):
    imp = pd.DataFrame({"Variable": nombresArboles,
                        "Importancia": modelosEntrenados[nombre].featureImportances.toArray()}) \
            .sort_values("Importancia", ascending=False)
    sns.barplot(data=imp, x="Importancia", y="Variable", color="steelblue", ax=eje)
    eje.set_title(f"Importancia de variables: {nombre}")
    print(nombre, dict(zip(imp["Variable"], imp["Importancia"].round(3))))
plt.tight_layout()
plt.show()

**Lectura.** Los dos modelos coinciden en las variables principales, aunque no en el orden:
- **GBT ajustado:** edad (18.6 %), educación (15.0 %), ocupación (13.2 %), relación (11.5 %), pérdida de capital (11.3 %) y estado civil (9.4 %).
- **Random Forest:** estado civil (21.6 %), educación (17.7 %), relación (17.6 %) y ganancia de capital (15.4 %).

La diferencia en el orden se explica porque estado civil, relación y edad están muy ligados entre sí, y cada modelo reparte la importancia de esa información compartida de forma distinta. `Sexo`, `Raza` y `PaisOrigen` pesan menos del 1 %. En el caso de `Sexo` no es porque no tenga relación con el ingreso, sino porque `Relacion` (`Husband` / `Wife`) ya contiene esa información.

### 6.8 Desempeño por grupo (sexo y raza)
`Sexo` y `Raza` son atributos sensibles. Se revisa si el modelo final se equivoca de forma distinta según el grupo: qué porcentaje gana >50K en realidad, qué porcentaje predice el modelo, y su precisión, recall y tasa de falsos positivos en cada grupo.

In [ ]:
def desempenoPorGrupo(pred, columna):
    pdg = (pred.groupBy(columna).agg(
              F.count("*").alias("Personas"),
              F.round(F.avg("Etiqueta")*100, 1).alias("Pct_real_mayor50K"),
              F.round(F.avg("prediction")*100, 1).alias("Pct_predicho_mayor50K"),
              F.sum(((F.col("Etiqueta") == 1) & (F.col("prediction") == 1)).cast("int")).alias("VP"),
              F.sum(((F.col("Etiqueta") == 0) & (F.col("prediction") == 1)).cast("int")).alias("FP"),
              F.sum(((F.col("Etiqueta") == 1) & (F.col("prediction") == 0)).cast("int")).alias("FN"),
              F.sum(((F.col("Etiqueta") == 0) & (F.col("prediction") == 0)).cast("int")).alias("VN"))
           .toPandas())
    pdg["Precision"] = (pdg["VP"]/(pdg["VP"] + pdg["FP"])).round(3)
    pdg["Recall"] = (pdg["VP"]/(pdg["VP"] + pdg["FN"])).round(3)
    pdg["Tasa_FP"] = (pdg["FP"]/(pdg["FP"] + pdg["VN"])).round(3)
    return pdg.sort_values("Personas", ascending=False).reset_index(drop=True)

pdSexoModelo = desempenoPorGrupo(predicciones["GBT ajustado"], "Sexo")
pdRazaModelo = desempenoPorGrupo(predicciones["GBT ajustado"], "Raza")
print(pdSexoModelo)
print(pdRazaModelo)

pdGraf = pdSexoModelo.melt(id_vars="Sexo", value_vars=["Precision", "Recall", "Tasa_FP"], var_name="Métrica", value_name="Valor")
plt.figure(figsize=(8, 4))
sns.barplot(data=pdGraf, x="Métrica", y="Valor", hue="Sexo")
plt.title("GBT ajustado: desempeño por sexo (prueba)")
plt.show()

**Lectura.** 
- **Precisión:** es casi igual para hombres y mujeres (0.634 y 0.623). Cuando el modelo dice `>50K`, acierta igual en los dos grupos.
- **Recall:** es **más bajo en mujeres (0.728) que en hombres (0.824)**. El modelo deja sin detectar al 27 % de las mujeres con ingresos altos, frente al 18 % de los hombres.
- **Falsos positivos:** son mucho más frecuentes en hombres (0.21 frente a 0.05). El modelo marca como >50K a muchos hombres que no lo son.
- **Raza:** el recall va de 0.817 (`White`) a 0.725 (`Asian-Pac-Islander`). Los grupos `Amer-Indian-Eskimo` y `Other` tienen menos de 100 personas en prueba, así que sus cifras no son confiables.

El modelo reproduce la brecha del censo de 1994. Quitar la variable `Sexo` no basta, porque `Relacion` (`Husband` / `Wife`) la contiene. Por eso, si se usa para decisiones sobre personas, hay que medir estas diferencias y corregirlas (sección 7).

### 6.9 Análisis de resultados
Resultados en el conjunto de prueba (9552 personas, 2282 con >50K), ordenados por la posición promedio en AUC-ROC, AUC-PR y F1:

| Modelo | AUC-ROC | AUC-PR | Accuracy | Precisión | Recall | F1 | Posición promedio |
|---|---|---|---|---|---|---|---|
| **GBT ajustado (umbral 0.30)** | **0.9173** | **0.7880** | 84.23 % | 0.633 | **0.810** | **0.711** | **1.0** |
| GBT (base) | 0.9171 | 0.7876 | **85.94 %** | 0.744 | 0.628 | 0.681 | 2.0 |
| Random Forest | 0.9132 | 0.7816 | 85.61 % | **0.763** | 0.577 | 0.657 | 3.67 |
| Perceptrón multicapa | 0.9040 | 0.7479 | 84.80 % | 0.712 | 0.612 | 0.658 | 4.33 |
| Árbol de decisión | 0.8878 | 0.7521 | 84.89 % | 0.702 | 0.638 | 0.669 | 4.67 |
| Regresión logística | 0.9008 | 0.7342 | 84.50 % | 0.706 | 0.602 | 0.650 | 6.00 |
| Linear SVC | 0.8980 | 0.7338 | 84.58 % | 0.709 | 0.601 | 0.651 | 6.33 |
| Naive Bayes | 0.8655 | 0.6540 | 76.64 % | 0.507 | 0.783 | 0.616 | 8.0 |

*Línea base (predecir siempre `<=50K`): 76.11 % de accuracy.*

**Lo que muestran los números**
1. **Todos aprenden algo real, pero no todos por igual.** Todos los modelos superan el 0.5 de AUC del azar por mucho (0.87 a 0.92). En accuracy, seis superan la línea base por 8 a 10 puntos, mientras que Naive Bayes la supera por apenas medio punto.
2. **Los conjuntos de árboles son los mejores.** GBT y Random Forest encabezan la tabla, y el bootstrap confirma que GBT supera a Random Forest y a la regresión logística. Combinar muchos árboles captura las interacciones que se vieron en el EDA (edad × educación, estado civil × sexo), y además reduce el sobreajuste de un único árbol: el árbol solo queda 0.03 por debajo en AUC.
3. **Los modelos lineales y la red neuronal quedan en un punto medio.** La regresión logística y Linear SVC son prácticamente iguales (0.901 y 0.898). El perceptrón multicapa, que puede aprender relaciones no lineales, mejora un poco (0.904), pero no alcanza a los árboles.
4. **Naive Bayes es el más débil (AUC 0.866).** Supone que las variables son independientes, y aquí no lo son: estado civil, relación y sexo dicen casi lo mismo. Al contar varias veces la misma evidencia, exagera la probabilidad de >50K, y por eso tiene mucho recall (0.78) pero precisión de 0.51.
5. **El mayor aporte del ajuste vino del umbral, no de los hiperparámetros.** La validación cruzada eligió árboles de profundidad 4 en lugar de 5 y 100 árboles en lugar de 50, pero el AUC casi no cambió (+0.0002, no significativo). En cambio, mover el umbral de 0.5 a 0.30 subió el F1 de 0.684 a 0.711 y el recall de 0.63 a 0.81. Con clases desbalanceadas, el umbral de 0.5 no es necesariamente el mejor.
6. **Costo:** Naive Bayes y el árbol de decisión entrenan en 1 a 2 segundos; GBT y Random Forest tardan alrededor de 10 veces más. La validación cruzada de GBT fue la parte más costosa del cuaderno. Aun así, todos los tiempos son manejables para 39 mil registros en el clúster, y la mejora de GBT justifica su costo.

**Modelo elegido: GBT ajustado (100 árboles de profundidad 4, umbral 0.30).** Es el primero en AUC-ROC, AUC-PR y F1. Si el uso exige máxima precisión, el mismo modelo sirve con un umbral más alto (sección 7). Si se necesita un modelo simple de explicar, la regresión logística pierde 0.017 de AUC y entrena en pocos segundos.

# **7. Conclusiones y recomendaciones**
### 7.1 Cumplimiento de los objetivos
| Objetivo específico | Resultado |
|---|---|
| 1. Cargar el dataset en HDFS y leerlo con PySpark | `adult.data` y `adult.test` en `/Formatos1055/BuenoLuis/Salario`, leídos con un esquema explícito que resuelve el formato de UCI (48 842 registros) |
| 2. EDA | Estructura, estadísticas, 8 gráficas de relación con el ingreso, correlaciones, interacción edad × educación y tabla de hallazgos |
| 3. Diagnóstico de calidad con evidencia | 7 problemas identificados, cada uno con una gráfica o tabla y una decisión justificada (3.6) |
| 4. Tratamiento y pipeline | Limpieza aplicada (48 790 registros), división 80/20 reproducible y `Pipeline` de 12 etapas ajustado solo con entrenamiento |
| 5. Modelos y ajuste | 7 modelos de MLlib; `CrossValidator` de 3 pliegues para GBT y regresión logística; umbral elegido en validación |
| 6. Comparación | 8 métricas, ranking, curvas ROC y PR, matrices de confusión, intervalos de confianza por bootstrap, costo, importancia de variables y análisis por grupo |
| 7. Implantación y conclusiones | `PipelineModel` guardado en HDFS y verificado; conclusiones y recomendaciones en esta sección |

### 7.2 Criterios de éxito
| Criterio | Meta | Resultado del modelo final | ¿Cumple? |
|---|---|---|---|
| Superar la línea base | > 76.1 % de accuracy | 84.2 % | Sí |
| Capacidad de ordenar | AUC-ROC ≥ 0.90 | 0.917 | Sí |
| Equilibrio en la clase `>50K` | F1 ≥ 0.65 | 0.711 | Sí |
| Costo | Segundos o pocos minutos | Entrenamiento en segundos; validación cruzada en pocos minutos | Sí |

### 7.3 Conclusiones
1. **Sí se puede estimar con buena confiabilidad si una persona gana más de 50K usando solo datos demográficos y laborales.** Si se toma al azar una persona que gana más de 50K y otra que no, el mejor modelo le asigna más probabilidad a la primera en el 91.7 % de los casos (AUC 0.917) y encuentra al 81 % de quienes ganan más de 50K. Aun así, no es infalible: de las personas que marca como >50K, el 37 % no lo es.
2. **La preparación de los datos se apoyó en evidencia, no en reglas fijas.** Los faltantes se conservaron como categoría porque tienen un patrón propio (9.4 % frente a 24.8 %). Los "atípicos" de la regla IQR no se borraron porque eran valores válidos, y además informativos en el caso del capital. `PesoFinal` se eliminó porque no tiene relación con el ingreso (correlación −0.006).
3. **Lo que más se asocia a ganar más de 50K** es la combinación de edad (45–54 años), educación universitaria o de posgrado, ocupación directiva o profesional, vivir con el cónyuge y tener ganancias de capital. Estos factores interactúan entre sí, y por eso los modelos de árboles superan a los lineales.
4. **GBT es el mejor modelo, y la diferencia frente a los demás se confirmó con bootstrap.** Random Forest queda cerca (−0.004 de AUC); los modelos lineales y la red neuronal, un escalón abajo; Naive Bayes, último, porque su supuesto de independencia no se cumple en estos datos.
5. **El umbral de decisión importó más que los hiperparámetros.** La validación cruzada casi no cambió el AUC, mientras que elegir el umbral con datos de validación subió el recall de 63 % a 81 % y el F1 de 0.68 a 0.71. Con clases desbalanceadas, el umbral de 0.5 casi nunca es el adecuado.
6. **Spark tiene detalles que cambian los resultados si se ignoran**:
   - Materializar el DataFrame antes de `randomSplit`; si no, las divisiones cambian entre acciones.
   - Ajustar el pipeline solo con entrenamiento.
   - Calcular el AUC de Naive Bayes con la probabilidad y no con `rawPrediction`.
   - Usar `thresholds` para guardar el umbral dentro del modelo.
   - Guardar el `PipelineModel` completo en HDFS, para que cualquier aplicación del clúster pueda predecir sobre datos crudos.
7. **El modelo hereda las brechas del censo de 1994.** Detecta peor a las mujeres con ingresos altos (recall 0.73 frente a 0.82) y marca de más a los hombres (tasa de falsos positivos 0.21 frente a 0.05).

### 7.4 Recomendaciones de uso
El mismo modelo sirve para distintos usos cambiando solo el umbral. Las cifras son las del conjunto de validación (5.3):

| Uso | Qué importa | Umbral sugerido | Precisión | Recall |
|---|---|---|---|---|
| **Mercadeo o productos bancarios** para personas de ingresos altos | Que cada oferta llegue a quien sí gana >50K (pocas ofertas desperdiciadas) | 0.60 | 0.80 | 0.53 |
| **Uso general** (equilibrio) | Balance entre encontrar y acertar | **0.30** | 0.62 | 0.80 |
| **Focalización de subsidios** (excluir a quien gana >50K) | No dejar pasar a nadie con ingresos altos | 0.25 | 0.59 | 0.84 |

- **No usar el modelo como única base para decisiones sobre personas** (créditos, empleo, subsidios) sin revisión humana. Con umbral 0.30, el 37 % de quienes marca como >50K no lo son.
- **Antes de usarlo en decisiones reales, corregir la diferencia por sexo.** Por ejemplo, con umbrales por grupo que igualen el recall o con métodos de mitigación de sesgo. Después hay que volver a medir la equidad, también por raza, con más datos en los grupos pequeños.
- **Volver a entrenarlo con datos recientes.** Los datos son de 1994: 50 000 dólares de entonces no equivalen al mismo nivel de ingreso hoy, y la estructura del empleo cambió.

### 7.5 Recomendaciones técnicas y trabajo futuro
- **Ampliar la grilla de GBT:** el mejor valor de `maxIter` quedó en el borde (100). Conviene probar 150–300 árboles con `stepSize` menor, y también `subsamplingRate`.
- **Ajustar Random Forest y el perceptrón multicapa con `CrossValidator`**, usando AUC-PR como métrica, que es más sensible a la clase minoritaria.
- **Probar pesos por clase (`weightCol`)** como alternativa al cambio de umbral, y comparar los dos enfoques.
- **Crear nuevas variables**, como grupos de edad, interacciones edad × educación o un indicador de ganancias de capital, sobre todo para los modelos lineales.
- **Para volúmenes mucho mayores**, guardar los datos limpios en Parquet en HDFS en lugar de releer los CSV, y ajustar `spark.sql.shuffle.partitions` al tamaño del clúster.

### 7.6 Limitaciones
- **Una sola división entrenamiento/prueba:** los intervalos de bootstrap miden la incertidumbre de la prueba, pero no la de la división.
- **Grillas pequeñas en la validación cruzada:** se mantuvieron pequeñas para que el cuaderno corra en tiempos razonables en el clúster compartido.
- **Datos de un solo país y de 1994:** las conclusiones sobre factores de ingreso no se pueden trasladar directamente a otro contexto, como Colombia hoy.

# **8. English summary**
**Goal.** Predict whether a person earns more than USD 50K per year from 1994 US Census data (UCI *Census Income*, 48,842 records), using PySpark, Spark MLlib and Hadoop HDFS on the course cluster.

**Data preparation.** Several data quality issues were found and handled:
- 52 duplicates were removed.
- Two columns were dropped: `fnlwgt`, a sampling weight with no relation to income (r = −0.006), and `education`, a 1-to-1 duplicate of `education-num`.
- The 3,620 records with missing values (7.4 %) were kept as an `Unknown` category, because they earn differently (9.4 % vs 24.8 % above 50K).
- Rare categories were grouped, and capital variables were log-transformed (skewness 11.9 → 3.1).

A 12-stage feature `Pipeline` (indexing, one-hot encoding, assembling, scaling) was fitted on training data only.

**Models.** Seven MLlib classifiers were trained. Gradient-boosted trees and logistic regression were then tuned with 3-fold `CrossValidator`, and the decision threshold was chosen on a separate validation set. The test set was used only for the final evaluation.

**Results.** The **tuned GBT** (100 trees, depth 4, threshold 0.30) is the best model: **ROC-AUC 0.917, PR-AUC 0.788, F1 0.711, recall 0.81** on the test set. Bootstrap tests (500 resamples) confirm that it beats random forest (+0.004 AUC) and logistic regression (+0.017 AUC). Most of the gain from tuning came from the threshold, not from the hyperparameters. The final pipeline was saved to HDFS and reloaded with identical predictions.

**Key findings.** Age, education, occupation, household role and capital gains drive income, and they interact with each other: education matters much more at ages 45–54 than at 17–24. This is why tree ensembles outperform linear models.

**Recommendations.**
- Choose the threshold according to the use case: 0.60 for high-precision marketing, 0.30 for a balanced use, 0.25 for subsidy targeting.
- Do not use the model alone for decisions about people. It has lower recall for women (0.73 vs 0.82 for men), and it was trained on 1994 data.
- Next steps: wider hyperparameter grids, class weights, and new features.

## Repositorio
El cuaderno, el README del taller y las instrucciones para obtener los datos están en:
**https://github.com/Luis-bu/Procesamiento_Datos/tree/main/Taller_Salarios_MLlib**

## Referencias
- Kohavi, R. (1996). *Census Income* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5GP7S
- Kohavi, R. (1996). Scaling Up the Accuracy of Naive-Bayes Classifiers: a Decision-Tree Hybrid. *Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD-96)*.
- Apache Spark. *MLlib: Classification and regression*. https://spark.apache.org/docs/3.5.0/ml-classification-regression.html
- Apache Spark. *ML Tuning: model selection and hyperparameter tuning*. https://spark.apache.org/docs/3.5.0/ml-tuning.html
- Chapman, P. et al. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS.

### Liberación de los recursos del clúster

In [ ]:
sparkLuis.stop()